# 📊 Análisis Exploratorio de Momentum en Criptoactivos

Este notebook implementa un análisis cuantitativo completo para evaluar la estructura de momentum en activos digitales (BTC, ETH) a través de diferentes horizontes temporales.

## 🎯 Objetivos:
- Medir **autocorrelación** en retornos
- Evaluar **persistencia de momentum** (hit rate)
- Analizar **clustering de volatilidad**
- Cuantificar **fuerza direccional** y rachas
- Determinar si existen patrones explotables para trading

## 1️⃣ Configuración Inicial e Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
import ccxt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from datetime import datetime, timedelta
import pandas_ta as ta
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Librerías importadas correctamente")
print(f"✅ pandas-ta versión: {ta.version}")

✅ Librerías importadas correctamente
✅ pandas-ta versión: 0.4.71b0


### 🎯 Opción Alternativa: Usar Módulos Modularizados

El código de este notebook ha sido modularizado en `src/trading_strategy/` para facilitar debug, testing y reutilización:

```python
# Importar módulos
from trading_strategy import (
    load_saved_data,
    fetch_ohlcv_data,
    calculate_returns_and_momentum,
    calculate_indicator_and_signals,
    combine_indicator_signals,
    backtest_strategy,
    strategy_grid_search,
    visualize_grid_search_results,
    export_best_strategies
)

# Uso rápido
data = load_saved_data('btcusdt', ['1h'])
df = calculate_returns_and_momentum(data['1h'], compute_indicators=False)
results = strategy_grid_search(df, configs, ticker='BTCUSDT', timeframe='1h')
```

**Ventajas de usar los módulos:**
- ✅ Mejor debug (stack traces claros)
- ✅ Testing unitario más fácil
- ✅ Reutilización en otros proyectos
- ✅ Separación de responsabilidades
- ✅ Versionamiento independiente

Ver `examples/basic_usage.py` para ejemplo completo o seguir usando las funciones definidas en el notebook.

## 2️⃣ Descarga de Datos Multi-Horizonte

In [2]:
def download_multi_timeframe_data(ticker, start_date='2020-01-01', save_data=True):
    """
    Descarga datos en múltiples horizontes temporales usando CCXT y Binance
    
    Parameters:
    -----------
    ticker : str
        Símbolo del par en formato 'BTC/USDT', 'ETH/USDT', etc.
    start_date : str
        Fecha de inicio en formato 'YYYY-MM-DD'
    save_data : bool
        Si True, guarda los datos en formato Parquet en ../data/market
    """
    import os
    
    print(f"📊 Descargando datos de {ticker} desde Binance...")
    
    # Inicializar exchange
    exchange = ccxt.binance({
        'enableRateLimit': True,
        'options': {'defaultType': 'spot'}
    })
    
    # Convertir ticker a formato CCXT si viene en formato tradicional
    if '-' in ticker:
        # Formato BTC-USD -> BTC/USDT
        ticker = ticker.replace('-USD', '/USDT').replace('-', '/')
    
    # Convertir fecha a timestamp
    start_timestamp = int(datetime.strptime(start_date, '%Y-%m-%d').timestamp() * 1000)
    
    data = {}
    timeframes = {
        '1h': '1h',
        '4h': '4h',
        '1d': '1d'
    }
    
    # Crear directorio si no existe
    if save_data:
        market_dir = '../data/market'
        os.makedirs(market_dir, exist_ok=True)
    
    # Nombre limpio del ticker para archivos
    ticker_clean = ticker.replace('/', '').replace('-', '').lower()
    
    for tf_name, tf_ccxt in timeframes.items():
        print(f"  Descargando {tf_name}...", end='')
        
        try:
            # Descargar todos los datos desde start_date
            all_ohlcv = []
            current_timestamp = start_timestamp
            
            while True:
                ohlcv = exchange.fetch_ohlcv(
                    ticker, 
                    timeframe=tf_ccxt, 
                    since=current_timestamp,
                    limit=1000  # Máximo por request
                )
                
                if not ohlcv:
                    break
                    
                all_ohlcv.extend(ohlcv)
                
                # Actualizar timestamp para siguiente batch
                current_timestamp = ohlcv[-1][0] + 1
                
                # Si llegamos al presente, terminar
                if ohlcv[-1][0] >= exchange.milliseconds():
                    break
            
            # Convertir a DataFrame
            df = pd.DataFrame(
                all_ohlcv,
                columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']
            )
            
            # Convertir timestamp a datetime
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.set_index('timestamp', inplace=True)
            
            # Asegurar que los datos sean numéricos
            for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # Eliminar duplicados y valores nulos
            df = df[~df.index.duplicated(keep='first')]
            df = df.dropna()
            
            data[tf_name] = df
            print(f" ✓ {len(df)} velas")
            
            # Guardar en formato Parquet
            if save_data:
                date_start = df.index.min().strftime('%Y%m%d')
                date_end = df.index.max().strftime('%Y%m%d')
                parquet_filename = f'{ticker_clean}_{tf_name}_{date_start}_{date_end}.parquet'
                parquet_filepath = os.path.join(market_dir, parquet_filename)
                
                df.to_parquet(parquet_filepath, compression='snappy', index=True)
                print(f"    💾 Guardado: {parquet_filepath}")
            
        except Exception as e:
            print(f" ✗ Error: {str(e)}")
            continue
    
    if data:
        print(f"\n✓ Datos descargados exitosamente")
        print(f"  Rango temporal: {min(df.index.min() for df in data.values())} a {max(df.index.max() for df in data.values())}")
        if save_data:
            print(f"  📁 Archivos guardados en: {market_dir}/")
    else:
        print("\n✗ No se pudieron descargar datos")
    
    return data


def load_saved_data(ticker, timeframes=['1h', '4h', '1d']):
    """
    Carga datos guardados en formato Parquet desde ../data/market
    
    Parameters:
    -----------
    ticker : str
        Símbolo del par en formato 'BTC/USDT', 'ETH/USDT', etc.
    timeframes : list
        Lista de timeframes a cargar ['1h', '4h', '1d']
    
    Returns:
    --------
    dict : Diccionario con DataFrames por timeframe
    """
    import os
    import glob
    
    print(f"📂 Cargando datos guardados de {ticker}...")
    
    # Nombre limpio del ticker
    ticker_clean = ticker.replace('/', '').replace('-', '').lower()
    market_dir = '../data/market'
    
    if not os.path.exists(market_dir):
        print(f"❌ Directorio {market_dir} no existe")
        return None
    
    data = {}
    
    for tf in timeframes:
        # Buscar archivo más reciente para este ticker y timeframe
        pattern = f'{ticker_clean}_{tf}_*.parquet'
        files = glob.glob(os.path.join(market_dir, pattern))
        
        if files:
            # Ordenar por fecha de modificación (más reciente primero)
            latest_file = max(files, key=os.path.getmtime)
            
            try:
                df = pd.read_parquet(latest_file)
                data[tf] = df
                print(f"  ✓ {tf}: {len(df)} velas | {df.index[0]} a {df.index[-1]}")
                print(f"    📄 Archivo: {os.path.basename(latest_file)}")
            except Exception as e:
                print(f"  ✗ Error cargando {tf}: {str(e)}")
        else:
            print(f"  ⚠️  {tf}: No se encontró archivo guardado")
    
    if data:
        print(f"\n✓ Datos cargados exitosamente desde disco")
    else:
        print(f"\n⚠️  No se encontraron datos guardados para {ticker}")
        print(f"   Usa download_multi_timeframe_data() con save_data=True para descargar")
    
    return data if data else None


# Probar la función
#data_test = download_multi_timeframe_data('BTC/USDT', start_date='2023-01-01', save_data=True)
#if data_test:
#    print(f"\n📋 Resumen:")
#    for tf, df in data_test.items():
#        print(f"  {tf}: {len(df):,} observaciones | {df.index[0]} a {df.index[-1]}")

# Para cargar datos guardados:
# data_loaded = load_saved_data('BTC/USDT')

print("✅ Funciones de descarga y carga con Parquet listas")

✅ Funciones de descarga y carga con Parquet listas


## 3️⃣ Cálculo de Retornos y Momentum

In [3]:
def calculate_returns_and_momentum(df, lookback_periods=[5, 10, 20, 30, 60], compute_indicators=True):
    """
    Calcula retornos y múltiples indicadores de momentum usando pandas-ta
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame con datos OHLCV
    lookback_periods : list
        Lista de períodos para indicadores que los soportan
    compute_indicators : bool or str
        - True: Calcula todo (retornos + indicadores)
        - False: Solo retornos básicos
        - 'returns_only': Solo retornos básicos (alias de False)
        - 'full': Retornos + indicadores (alias de True)
    
    Indicadores implementados (36 total):
    - Oscillators: RSI, ROC, MOM, Stochastic, CCI, Williams %R, CMO, UO
    - Trend: MACD, ADX, ER, Slope, Trix, AO, APO, Coppock, DPO
    - Momentum Advanced: Bias, BOP, CFO, CG, CTI, Inertia, RSX
    - Compression: Squeeze, Squeeze Pro, Bollinger Bands
    - Cycle: Reflex, Trendflex, Fisher
    - Volatility: RVI, Elder's Thermometer, ATR, MAD, STDEV, Variance, Entropy
    """
    df = df.copy()
    
    # ========== RETORNOS BÁSICOS (SIEMPRE SE CALCULAN) ==========
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['candle_rtn'] = (df['Close'] - df['Open']) / df['Open']  # Retorno intra-vela
    
    # Si solo queremos retornos, retornar aquí
    if compute_indicators is False or compute_indicators == 'returns_only':
        return df
    
    # ========== INDICADORES PANDAS-TA ==========
    
    # 1. RSI (Relative Strength Index) - Múltiples períodos
    for period in lookback_periods:
        df[f'rsi_{period}'] = ta.rsi(df['Close'], length=period)
        # Señal: RSI > 70 = sobrecompra (bajista), RSI < 30 = sobreventa (alcista), resto = neutral
        df[f'rsi_signal_{period}'] = np.where(df[f'rsi_{period}'] > 70, -1, 
                                              np.where(df[f'rsi_{period}'] < 30, 1, 0))
    
    # 2. ROC (Rate of Change) - Similar al cálculo original pero optimizado
    for period in lookback_periods:
        df[f'roc_{period}'] = ta.roc(df['Close'], length=period)
        df[f'roc_signal_{period}'] = np.sign(df[f'roc_{period}'])
    
    # 3. MOM (Momentum) - Diferencia absoluta de precios
    for period in lookback_periods:
        df[f'mom_{period}'] = ta.mom(df['Close'], length=period)
        df[f'mom_signal_{period}'] = np.sign(df[f'mom_{period}'])
    
    # 4. MACD (Moving Average Convergence Divergence)
    # Usamos el período más común: 12, 26, 9
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    if macd is not None:
        df['macd'] = macd['MACD_12_26_9']
        df['macd_signal'] = macd['MACDs_12_26_9']
        df['macd_hist'] = macd['MACDh_12_26_9']
        df['macd_cross'] = np.where(df['macd'] > df['macd_signal'], 1, -1)
    
    # 5. Stochastic Oscillator (14, 3, 3)
    stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=14, d=3)
    if stoch is not None:
        df['stoch_k'] = stoch[f'STOCHk_14_3_3']
        df['stoch_d'] = stoch[f'STOCHd_14_3_3']
        # Señal: K > D = alcista, K < D = bajista
        df['stoch_signal'] = np.where(df['stoch_k'] > df['stoch_d'], 1, -1)
    
    # 6. CCI (Commodity Channel Index) - Períodos principales
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            df[f'cci_{period}'] = ta.cci(df['High'], df['Low'], df['Close'], length=period)
            # CCI > 0 = momentum alcista, CCI < 0 = momentum bajista
            df[f'cci_signal_{period}'] = np.where(df[f'cci_{period}'] > 0, 1, -1)
    
    # 7. Williams %R - Oscilador de momentum inverso
    for period in [14, 20, 30]:
        if period in lookback_periods or period in [14, 20]:
            willr = ta.willr(df['High'], df['Low'], df['Close'], length=period)
            if willr is not None:
                df[f'willr_{period}'] = willr
                # WillR > -20 = sobrecompra (bajista), WillR < -80 = sobreventa (alcista), resto = neutral
                df[f'willr_signal_{period}'] = np.where(df[f'willr_{period}'] > -20, -1,
                                                        np.where(df[f'willr_{period}'] < -80, 1, 0))
    
    # ========== OSCILADORES AVANZADOS ==========
    
    # 8. Awesome Oscillator (AO) - Diferencia de medias móviles de puntos medios
    ao = ta.ao(df['High'], df['Low'], fast=5, slow=34)
    if ao is not None:
        df['ao'] = ao
        df['ao_signal'] = np.where(df['ao'] > 0, 1, -1)
    
    # 9. Absolute Price Oscillator (APO) - Diferencia absoluta de EMAs
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            apo = ta.apo(df['Close'], fast=period, slow=period*2)
            if apo is not None:
                df[f'apo_{period}'] = apo
                df[f'apo_signal_{period}'] = np.sign(df[f'apo_{period}'])
    
    # 10. Bias - Desviación porcentual del precio respecto a su media
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            bias = ta.bias(df['Close'], length=period)
            if bias is not None:
                df[f'bias_{period}'] = bias
                df[f'bias_signal_{period}'] = np.where(df[f'bias_{period}'] > 0, 1, -1)
    
    # 11. Balance of Power (BOP) - Fuerza relativa compradores vs vendedores
    bop = ta.bop(df['Open'], df['High'], df['Low'], df['Close'])
    if bop is not None:
        df['bop'] = bop
        df['bop_signal'] = np.where(df['bop'] > 0, 1, -1)
    
    # 12. Chande Forecast Oscillator (CFO) - Predicción de tendencia
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            cfo = ta.cfo(df['Close'], length=period)
            if cfo is not None:
                df[f'cfo_{period}'] = cfo
                df[f'cfo_signal_{period}'] = np.where(df[f'cfo_{period}'] > 0, 1, -1)
    
    # 13. Center of Gravity (CG) - Centro de gravedad del precio
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            cg = ta.cg(df['Close'], length=period)
            if cg is not None:
                df[f'cg_{period}'] = cg
                df[f'cg_signal_{period}'] = np.sign(df[f'cg_{period}'])
    
    # 14. Chande Momentum Oscillator (CMO) - Momentum sin límites
    for period in [14, 20]:
        if period in lookback_periods or period == 14:
            cmo = ta.cmo(df['Close'], length=period)
            if cmo is not None:
                df[f'cmo_{period}'] = cmo
                df[f'cmo_signal_{period}'] = np.where(df[f'cmo_{period}'] > 0, 1, -1)
    
    # 15. Coppock Curve - Oscilador de largo plazo
    coppock = ta.coppock(df['Close'])
    if coppock is not None:
        df['coppock'] = coppock
        df['coppock_signal'] = np.where(df['coppock'] > 0, 1, -1)
    
    # 16. Correlation Trend Indicator (CTI) - Correlación de precio con tendencia
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            cti = ta.cti(df['Close'], length=period)
            if cti is not None:
                df[f'cti_{period}'] = cti
                df[f'cti_signal_{period}'] = np.where(df[f'cti_{period}'] > 0, 1, -1)
    
    # ========== INDICADORES DIRECCIONALES ==========
    
    # 17. Directional Movement (DM) - ADX, +DI, -DI
    dm = ta.dm(df['High'], df['Low'], length=14)
    if dm is not None:
        df['dm_plus'] = dm['DMP_14']
        df['dm_minus'] = dm['DMN_14']
        df['dm_signal'] = np.where(df['dm_plus'] > df['dm_minus'], 1, -1)
    
    adx_result = ta.adx(df['High'], df['Low'], df['Close'], length=14)
    if adx_result is not None:
        df['adx'] = adx_result['ADX_14']
        df['adx_plus'] = adx_result['DMP_14']
        df['adx_minus'] = adx_result['DMN_14']
        df['adx_signal'] = np.where(df['adx_plus'] > df['adx_minus'], 1, -1)
    
    # 18. Efficiency Ratio (ER) - Eficiencia del movimiento
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            er = ta.er(df['Close'], length=period)
            if er is not None:
                df[f'er_{period}'] = er
                # ER > 0.3 = tendencia fuerte (alcista), ER < 0.3 = tendencia débil (bajista)
                df[f'er_signal_{period}'] = np.where(df[f'er_{period}'] > 0.3, 1, -1)
    
    # ========== INDICADORES DE AGOTAMIENTO Y TRANSFORMACIÓN ==========
    
    # 19. Fisher Transform - Transforma precios a distribución gaussiana
    fisher = ta.fisher(df['High'], df['Low'], length=9)
    if fisher is not None:
        df['fisher'] = fisher['FISHERT_9_1']
        df['fisher_signal_line'] = fisher['FISHERTs_9_1']
        df['fisher_signal'] = np.where(df['fisher'] > df['fisher_signal_line'], 1, -1)
    
    # 20. Inertia - Momentum con suavizado RVI
    for period in [14, 20]:
        if period in lookback_periods or period == 14:
            inertia = ta.inertia(df['Close'], df['High'], df['Low'], length=period)
            if inertia is not None:
                df[f'inertia_{period}'] = inertia
                # Inertia > 60 = alcista, Inertia < 40 = bajista, resto = neutral
                df[f'inertia_signal_{period}'] = np.where(df[f'inertia_{period}'] > 60, 1,
                                                          np.where(df[f'inertia_{period}'] < 40, -1, 0))
    
    # 21. Relative Strength Xtra (RSX) - RSI suavizado
    for period in [14, 20]:
        if period in lookback_periods or period == 14:
            rsx = ta.rsx(df['Close'], length=period)
            if rsx is not None:
                df[f'rsx_{period}'] = rsx
                # RSX > 70 = sobrecompra (bajista), RSX < 30 = sobreventa (alcista), resto = neutral
                df[f'rsx_signal_{period}'] = np.where(df[f'rsx_{period}'] > 70, -1,
                                                      np.where(df[f'rsx_{period}'] < 30, 1, 0))
    
    # 22. Slope - Pendiente de la regresión lineal
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            slope = ta.slope(df['Close'], length=period)
            if slope is not None:
                df[f'slope_{period}'] = slope
                df[f'slope_signal_{period}'] = np.sign(df[f'slope_{period}'])
    
    # 23. Squeeze - Indicador de Bollinger Bands y Keltner Channels
    squeeze = ta.squeeze(df['High'], df['Low'], df['Close'])
    if squeeze is not None:
        df['squeeze'] = squeeze['SQZ_20_2.0_20_1.5']
        df['squeeze_on'] = squeeze['SQZ_ON']
        df['squeeze_off'] = squeeze['SQZ_OFF']
        df['squeeze_signal'] = np.where(df['squeeze'] > 0, 1, -1)
    
    # 24. Squeeze Pro - Versión mejorada del Squeeze
    squeeze_pro = ta.squeeze_pro(df['High'], df['Low'], df['Close'])
    if squeeze_pro is not None:
        df['squeeze_pro'] = squeeze_pro['SQZPRO_20_2.0_20_2.0_1.5_1.0'] #SQZPRO_20_2.0_20_2.0_1.5_1.0
        df['squeeze_pro_signal'] = np.where(df['squeeze_pro'] > 0, 1, -1)
    
    # 25. Trix - Triple exponential derivative
    for period in [14, 20]:
        if period in lookback_periods or period == 14:
            trix = ta.trix(df['Close'], length=period, signal=9)
            if trix is not None:
                df[f'trix_{period}'] = trix[f'TRIX_{period}_9']
                df[f'trix_signal_{period}'] = np.where(df[f'trix_{period}'] > 0, 1, -1)
    
    # 26. Ultimate Oscillator (UO) - Combina 3 timeframes
    uo = ta.uo(df['High'], df['Low'], df['Close'])
    if uo is not None:
        df['uo'] = uo
        df['uo_signal'] = np.where(df['uo'] > 50, 1, -1)
    
    # ========== VOLATILIDAD Y DISPERSIÓN ==========
    
    # 27. Entropy - Medida de incertidumbre/aleatoriedad
    for period in [10, 20]:
        if period in lookback_periods or period == 10:
            entropy = ta.entropy(df['Close'], length=period)
            if entropy is not None:
                df[f'entropy_{period}'] = entropy
                # Entropy alta = mayor incertidumbre
    
    # 28. Mean Absolute Deviation (MAD) - Desviación absoluta media
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            mad = ta.mad(df['Close'], length=period)
            if mad is not None:
                df[f'mad_{period}'] = mad
    
    # 29. Rolling Standard Deviation (STDEV) - Ya calculado como volatility_20
    for period in [10, 20, 30]:
        if period in lookback_periods or period in [10, 20]:
            stdev = ta.stdev(df['Close'], length=period)
            if stdev is not None:
                df[f'stdev_{period}'] = stdev
    
    # 30. Rolling Variance - Varianza del precio
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            variance = ta.variance(df['Close'], length=period)
            if variance is not None:
                df[f'variance_{period}'] = variance
    
    # 31. Bollinger Bands - Bandas de volatilidad
    bbands = ta.bbands(df['Close'], length=20, std=2)
    if bbands is not None:
        df['bb_lower'] = bbands['BBL_20_2.0_2.0']
        df['bb_middle'] = bbands['BBM_20_2.0_2.0']
        df['bb_upper'] = bbands['BBU_20_2.0_2.0']
        df['bb_bandwidth'] = bbands['BBB_20_2.0_2.0']
        df['bb_percent'] = bbands['BBP_20_2.0_2.0']
        # Señal: precio cerca de banda inferior = alcista, superior = bajista
        df['bb_signal'] = np.where(df['bb_percent'] < 0.2, 1, np.where(df['bb_percent'] > 0.8, -1, 0))
    
    # 32. Relative Volatility Index (RVI) - RSI aplicado a volatilidad
    for period in [14, 20]:
        if period in lookback_periods or period == 14:
            rvi = ta.rvi(df['Close'], df['High'], df['Low'], length=period)
            if rvi is not None:
                df[f'rvi_{period}'] = rvi
                df[f'rvi_signal_{period}'] = np.where(df[f'rvi_{period}'] > 50, 1, -1)
    
    # 33. Elder's Thermometer - Volatilidad direccional
    thermo = ta.thermo(df['High'], df['Low'], length=20)
    if thermo is not None:
        df['thermo'] = thermo['THERMO_20_2_0.5']
        df['thermo_ma'] = thermo['THERMOma_20_2_0.5']
        df['thermo_signal'] = np.where(df['thermo'] > df['thermo_ma'], 1, -1)
    
    # ========== INDICADORES DE CICLO Y TENDENCIA ==========
    
    # 34. Reflex - Indicador de ciclo
    reflex = ta.reflex(df['Close'], length=20)
    if reflex is not None:
        df['reflex'] = reflex
        df['reflex_signal'] = np.where(df['reflex'] > 0, 1, -1)
    
    # 35. Trendflex - Separa tendencia de ciclo
    trendflex = ta.trendflex(df['Close'], length=20)
    if trendflex is not None:
        df['trendflex'] = trendflex
        df['trendflex_signal'] = np.where(df['trendflex'] > 0, 1, -1)
    
    # 36. Detrend Price Oscillator (DPO) - Elimina tendencia para mostrar ciclos
    for period in [20, 30]:
        if period in lookback_periods or period == 20:
            dpo = ta.dpo(df['Close'], length=period)
            if dpo is not None:
                df[f'dpo_{period}'] = dpo
                df[f'dpo_signal_{period}'] = np.where(df[f'dpo_{period}'] > 0, 1, -1)
    
    # ========== MOMENTUM TRADICIONAL (para comparación) ==========
    for period in lookback_periods:
        df[f'momTrad_{period}'] = df['Close'] / df['Close'].shift(period) - 1
        df[f'momTrad_signal_{period}'] = np.sign(df[f'momTrad_{period}'])
    
    # ========== VOLATILIDAD ==========
    df['volatility_20'] = df['returns'].rolling(20).std()
    
    # Volatilidad ATR (Average True Range) - mejor que std para trading
    atr = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    if atr is not None:
        df['atr_14'] = atr
    
    # Solo eliminar filas donde falten datos críticos
    return df.dropna(subset=['returns', 'log_returns'])

print("✅ Función de cálculo de retornos y momentum (pandas-ta) lista")

✅ Función de cálculo de retornos y momentum (pandas-ta) lista


### 📊 Indicadores de Momentum Implementados (36 Indicadores)

Con **pandas-ta** ahora calculamos una suite completa de indicadores profesionales organizados por categorías:

#### 🎯 Osciladores Clásicos (1-8)
1. **RSI**: Relative Strength Index - Oscilador 0-100 para sobrecompra/sobreventa
2. **ROC**: Rate of Change - Cambio porcentual del precio
3. **MOM**: Momentum - Diferencia absoluta de precios
4. **MACD**: Convergencia/divergencia de medias móviles
5. **Stochastic**: Momentum relativo al rango High-Low
6. **CCI**: Commodity Channel Index - Desviación del precio promedio
7. **Williams %R**: Oscilador inverso de momentum
8. **ATR**: Average True Range - Volatilidad real

#### 🌀 Osciladores Avanzados (9-16)
9. **Awesome Oscillator**: Diferencia de medias móviles de puntos medios
10. **APO**: Absolute Price Oscillator - Diferencia absoluta de EMAs
11. **Bias**: Desviación porcentual respecto a media móvil
12. **BOP**: Balance of Power - Fuerza compradores vs vendedores
13. **CFO**: Chande Forecast Oscillator - Predicción de tendencia
14. **CG**: Center of Gravity - Centro de gravedad del precio
15. **CMO**: Chande Momentum Oscillator - Momentum sin límites
16. **Coppock Curve**: Oscilador de largo plazo para cambios de tendencia

#### 📐 Indicadores Direccionales (17-22)
17. **ADX/DM**: Average Directional Index y Directional Movement
18. **Efficiency Ratio**: Eficiencia del movimiento direccional
19. **CTI**: Correlation Trend Indicator - Correlación con tendencia
20. **Fisher Transform**: Transforma precios a distribución gaussiana
21. **Inertia**: Momentum con suavizado RVI
22. **RSX**: Relative Strength Xtra - RSI suavizado

#### 📈 Indicadores de Tendencia (23-26)
23. **Slope**: Pendiente de regresión lineal
24. **Squeeze**: Indicador de compresión (Bollinger + Keltner)
25. **Squeeze Pro**: Versión mejorada del Squeeze
26. **Trix**: Triple exponential derivative

#### 🎲 Osciladores Compuestos (27-28)
27. **Ultimate Oscillator**: Combina 3 timeframes diferentes
28. **True Strength Index**: Momentum con doble suavizado

#### 📊 Volatilidad y Dispersión (29-33)
29. **Entropy**: Medida de incertidumbre/aleatoriedad
30. **MAD**: Mean Absolute Deviation - Desviación absoluta media
31. **STDEV**: Desviación estándar móvil
32. **Variance**: Varianza del precio
33. **Bollinger Bands**: Bandas de volatilidad con % y ancho

#### 🌡️ Volatilidad Relativa (34-35)
34. **RVI**: Relative Volatility Index - RSI aplicado a volatilidad
35. **Elder's Thermometer**: Volatilidad direccional

#### 🔄 Indicadores de Ciclo (36-38)
36. **Reflex**: Indicador de ciclo
37. **Trendflex**: Separa tendencia de ciclo
38. **DPO**: Detrend Price Oscillator - Muestra ciclos sin tendencia

Cada indicador genera señales direccionales (1=alcista, -1=bajista, 0=neutral) para facilitar el análisis de persistencia de momentum.

### 🔧 Modos de Cálculo

La función `calculate_returns_and_momentum()` soporta dos modos:

#### **Modo 1: Solo Retornos** (`compute_indicators=False`)
Calcula únicamente:
- `returns`: Retornos simples
- `log_returns`: Retornos logarítmicos  
- `candle_rtn`: Retorno intra-vela (Close - Open)

**Uso:** Grid Search (calcula indicadores dinámicamente según parámetros)

#### **Modo 2: Retornos + Indicadores** (`compute_indicators=True`, default)
Calcula retornos + 36 indicadores técnicos con señales

**Uso:** EDA exploratorio, análisis comparativo de indicadores

In [4]:
# Ejemplo de uso - Modo retornos básicos (rápido)
# df_basic = calculate_returns_and_momentum(data, compute_indicators=False)
# print(df_basic.columns)  # ['Open', 'High', 'Low', 'Close', 'Volume', 'returns', 'log_returns', 'candle_rtn']

# Ejemplo de uso - Modo completo con indicadores (default)
# df_full = calculate_returns_and_momentum(data, compute_indicators=True)
# print(df_full.shape)  # Columnas: ~120+ (OHLCV + retornos + 36 indicadores con señales)

print("💡 Tip para Grid Search:")
print("   - Usa compute_indicators=False para datos base")
print("   - El grid search calcula indicadores dinámicamente con parámetros específicos")
print("   - Más eficiente y flexible que pre-calcular todo")

💡 Tip para Grid Search:
   - Usa compute_indicators=False para datos base
   - El grid search calcula indicadores dinámicamente con parámetros específicos
   - Más eficiente y flexible que pre-calcular todo


## 4️⃣ Análisis de Autocorrelación

In [5]:
def analyze_autocorrelation(returns, max_lags=50):
    """
    Análisis completo de autocorrelación
    """
    results = {}
    
    # ACF y PACF
    acf_values = acf(returns.dropna(), nlags=max_lags, fft=True)
    pacf_values = pacf(returns.dropna(), nlags=max_lags)
    
    # Test de Ljung-Box (H0: no hay autocorrelación)
    lb_test = acorr_ljungbox(returns.dropna(), lags=min(20, len(returns)//5), return_df=True)
    
    # Autocorrelación de retornos absolutos y cuadrados (cluster de volatilidad)
    acf_abs = acf(np.abs(returns.dropna()), nlags=max_lags, fft=True)
    acf_sq = acf((returns.dropna())**2, nlags=max_lags, fft=True)
    
    results['acf'] = acf_values
    results['pacf'] = pacf_values
    results['ljung_box'] = lb_test
    results['acf_abs'] = acf_abs
    results['acf_squared'] = acf_sq
    
    return results

print("✅ Función de análisis de autocorrelación lista")

✅ Función de análisis de autocorrelación lista


## 5️⃣ Ratio de Persistencia (Hit Rate)

In [6]:
def momentum_persistence_analysis(df, lookback_periods=[5, 10, 20, 30], indicators='all'):
    """
    Mide si las señales de momentum/indicadores predicen dirección futura
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame con datos OHLCV e indicadores calculados
    lookback_periods : list
        Períodos a analizar
    indicators : str or list
        - 'all': Analiza todos los indicadores disponibles
        - 'traditional': Solo momentum tradicional
        - 'oscillators': RSI, Stochastic, CCI, Williams %R
        - 'trend': MACD, ADX, Slope, Trix
        - 'volatility': Bollinger Bands, ATR-based
        - list: Lista específica de nombres de indicadores (ej: ['rsi', 'macd', 'adx'])
    
    Returns:
    --------
    DataFrame con métricas de hit rate por indicador y período
    """
    results = []
    df = df.copy()
    df['future_ret'] = df['returns'].shift(-1)
    
    # Definir grupos de indicadores
    indicator_groups = {
        'traditional': ['momTrad'],
        'oscillators': ['rsi', 'roc', 'mom', 'stoch', 'cci', 'willr', 'cmo', 'rsx', 'uo', 'fisher'],
        'trend': ['macd', 'adx', 'slope', 'trix', 'ao', 'apo', 'coppock', 'dpo'],
        'momentum_advanced': ['bias', 'bop', 'cfo', 'cg', 'cti', 'er', 'inertia'],
        'compression': ['squeeze', 'squeeze_pro', 'bb'],
        'cycle': ['reflex', 'trendflex'],
        'volatility': ['rvi', 'thermo']
    }
    
    # Determinar qué indicadores analizar
    if indicators == 'all':
        indicators_to_analyze = [ind for group in indicator_groups.values() for ind in group]
    elif indicators == 'traditional':
        indicators_to_analyze = indicator_groups['traditional']
    elif indicators in indicator_groups:
        indicators_to_analyze = indicator_groups[indicators]
    elif isinstance(indicators, list):
        indicators_to_analyze = indicators
    else:
        indicators_to_analyze = indicator_groups['traditional']
    
    # Analizar cada indicador
    for indicator_base in indicators_to_analyze:
        # Buscar columnas de señales que coincidan con este indicador
        signal_cols = [col for col in df.columns if f'{indicator_base}_signal' in col or 
                      (indicator_base == 'macd' and col == 'macd_cross') or
                      (indicator_base == 'stoch' and col == 'stoch_signal') or
                      (indicator_base == 'fisher' and col == 'fisher_signal') or
                      (indicator_base == 'adx' and col == 'adx_signal') or
                      (indicator_base == 'dm' and col == 'dm_signal') or
                      (indicator_base == 'bb' and col == 'bb_signal') or
                      (indicator_base == 'squeeze' and col == 'squeeze_signal') or
                      (indicator_base == 'squeeze_pro' and col == 'squeeze_pro_signal') or
                      (indicator_base == 'ao' and col == 'ao_signal') or
                      (indicator_base == 'bop' and col == 'bop_signal') or
                      (indicator_base == 'coppock' and col == 'coppock_signal') or
                      (indicator_base == 'reflex' and col == 'reflex_signal') or
                      (indicator_base == 'trendflex' and col == 'trendflex_signal') or
                      (indicator_base == 'thermo' and col == 'thermo_signal')]
        
        for signal_col in signal_cols:
            # Extraer período si existe en el nombre de la columna
            period = None
            for p in lookback_periods:
                if f'_{p}' in signal_col:
                    period = p
                    break
            
            # Si no hay período específico o el indicador no usa períodos, usar 'N/A'
            if period is None:
                if any(x in signal_col for x in ['macd', 'stoch', 'fisher', 'ao', 'bop', 
                                                   'coppock', 'squeeze', 'uo', 'reflex', 
                                                   'trendflex', 'thermo', 'bb']):
                    period = 'default'
                else:
                    continue
            
            # Validar que la columna existe y tiene datos
            if signal_col not in df.columns:
                continue
            
            # Filtrar datos válidos
            valid = df[[signal_col, 'future_ret']].dropna()
            
            if len(valid) > 10:  # Mínimo 10 observaciones
                signal = valid[signal_col]
                future_ret = valid['future_ret']
                
                # Hit rate: % de veces que la señal coincide con el retorno futuro
                hit_rate = (signal * future_ret > 0).mean()
                
                # Retornos por tipo de señal
                long_mask = signal > 0
                short_mask = signal < 0
                neutral_mask = signal == 0
                
                avg_ret_long = future_ret[long_mask].mean() if long_mask.sum() > 0 else 0
                avg_ret_short = future_ret[short_mask].mean() if short_mask.sum() > 0 else 0
                avg_ret_neutral = future_ret[neutral_mask].mean() if neutral_mask.sum() > 0 else 0
                
                # Sharpe direccional
                sharpe_long = (future_ret[long_mask].mean() / future_ret[long_mask].std() 
                              if long_mask.sum() > 1 and future_ret[long_mask].std() > 0 else 0)
                sharpe_short = (-future_ret[short_mask].mean() / future_ret[short_mask].std() 
                               if short_mask.sum() > 1 and future_ret[short_mask].std() > 0 else 0)
                
                # Win rate (% de operaciones ganadoras)
                win_rate_long = (future_ret[long_mask] > 0).mean() if long_mask.sum() > 0 else 0
                win_rate_short = (future_ret[short_mask] < 0).mean() if short_mask.sum() > 0 else 0
                
                # Profit factor
                gross_profit_long = future_ret[long_mask][future_ret[long_mask] > 0].sum()
                gross_loss_long = abs(future_ret[long_mask][future_ret[long_mask] < 0].sum())
                profit_factor_long = gross_profit_long / gross_loss_long if gross_loss_long > 0 else 0
                
                gross_profit_short = abs(future_ret[short_mask][future_ret[short_mask] < 0].sum())
                gross_loss_short = future_ret[short_mask][future_ret[short_mask] > 0].sum()
                profit_factor_short = gross_profit_short / gross_loss_short if gross_loss_short > 0 else 0
                
                results.append({
                    'indicator': indicator_base,
                    'signal_column': signal_col,
                    'period': period,
                    'hit_rate': hit_rate,
                    'win_rate_long': win_rate_long,
                    'win_rate_short': win_rate_short,
                    'avg_return_long': avg_ret_long,
                    'avg_return_short': avg_ret_short,
                    'avg_return_neutral': avg_ret_neutral,
                    'sharpe_long': sharpe_long,
                    'sharpe_short': sharpe_short,
                    'profit_factor_long': profit_factor_long,
                    'profit_factor_short': profit_factor_short,
                    'n_long': long_mask.sum(),
                    'n_short': short_mask.sum(),
                    'n_neutral': neutral_mask.sum(),
                    'n_obs': len(valid)
                })
    
    results_df = pd.DataFrame(results)
    
    # Ordenar por hit rate descendente
    if not results_df.empty:
        results_df = results_df.sort_values('hit_rate', ascending=False)
    
    return results_df

print("✅ Función de análisis de persistencia mejorada lista")

✅ Función de análisis de persistencia mejorada lista


In [7]:
def compare_indicators_performance(df, top_n=20, min_observations=100):
    """
    Compara el rendimiento de todos los indicadores y muestra los mejores
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame con indicadores calculados
    top_n : int
        Número de mejores indicadores a mostrar
    min_observations : int
        Mínimo de observaciones para considerar un indicador válido
    
    Returns:
    --------
    DataFrame con ranking de indicadores por hit rate
    """
    print("\n" + "="*100)
    print("🏆 RANKING DE INDICADORES POR RENDIMIENTO")
    print("="*100 + "\n")
    
    # Analizar todos los indicadores
    all_results = momentum_persistence_analysis(df, indicators='all')
    
    if all_results.empty:
        print("❌ No se encontraron resultados")
        return None
    
    # Filtrar por mínimo de observaciones
    all_results = all_results[all_results['n_obs'] >= min_observations]
    
    # Mostrar top indicadores por hit rate
    print(f"📊 TOP {top_n} INDICADORES POR HIT RATE:")
    print("-" * 100)
    
    top_hit_rate = all_results.nlargest(top_n, 'hit_rate')
    
    for idx, row in top_hit_rate.iterrows():
        period_str = f"(periodo {row['period']})" if row['period'] != 'default' else ""
        print(f"\n{idx+1}. {row['indicator'].upper()} {period_str}")
        print(f"   Hit Rate: {row['hit_rate']:.1%} | "
              f"Win Long: {row['win_rate_long']:.1%} | "
              f"Win Short: {row['win_rate_short']:.1%}")
        print(f"   Sharpe Long: {row['sharpe_long']:.3f} | "
              f"Sharpe Short: {row['sharpe_short']:.3f}")
        print(f"   Avg Ret Long: {row['avg_return_long']*100:.4f}% | "
              f"Avg Ret Short: {row['avg_return_short']*100:.4f}%")
        print(f"   PF Long: {row['profit_factor_long']:.2f} | "
              f"PF Short: {row['profit_factor_short']:.2f}")
        print(f"   Señales: {row['n_long']} long, {row['n_short']} short, "
              f"{row['n_neutral']} neutral")
    
    # Mostrar estadísticas por categoría
    print("\n" + "="*100)
    print("📈 RENDIMIENTO PROMEDIO POR CATEGORÍA DE INDICADOR")
    print("="*100 + "\n")
    
    categories = {
        'Osciladores': ['rsi', 'roc', 'mom', 'cci', 'willr', 'cmo', 'rsx', 'uo'],
        'Tendencia': ['macd', 'adx', 'slope', 'trix', 'ao', 'apo', 'coppock', 'dpo'],
        'Momentum Avanzado': ['bias', 'bop', 'cfo', 'cg', 'cti', 'er', 'inertia'],
        'Compresión': ['squeeze', 'squeeze_pro', 'bb'],
        'Ciclo': ['reflex', 'trendflex', 'fisher'],
        'Volatilidad': ['rvi', 'thermo', 'stoch']
    }
    
    for category, indicators in categories.items():
        cat_results = all_results[all_results['indicator'].isin(indicators)]
        if not cat_results.empty:
            avg_hit_rate = cat_results['hit_rate'].mean()
            best_indicator = cat_results.nlargest(1, 'hit_rate').iloc[0]
            
            print(f"{category}:")
            print(f"   Hit Rate Promedio: {avg_hit_rate:.1%}")
            print(f"   Mejor: {best_indicator['indicator'].upper()} "
                  f"({best_indicator['hit_rate']:.1%})")
            print()
    
    print("="*100 + "\n")
    
    # Visualización
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Top indicadores por hit rate
    top_20 = all_results.nlargest(20, 'hit_rate')
    axes[0, 0].barh(range(len(top_20)), top_20['hit_rate'], color='steelblue')
    axes[0, 0].set_yticks(range(len(top_20)))
    axes[0, 0].set_yticklabels([f"{row['indicator']}_{row['period']}" 
                                 for _, row in top_20.iterrows()], fontsize=8)
    axes[0, 0].axvline(x=0.5, color='red', linestyle='--', label='Random (50%)')
    axes[0, 0].set_xlabel('Hit Rate')
    axes[0, 0].set_title('Top 20 Indicadores por Hit Rate')
    axes[0, 0].legend()
    axes[0, 0].invert_yaxis()
    
    # 2. Sharpe Long vs Sharpe Short
    axes[0, 1].scatter(all_results['sharpe_long'], all_results['sharpe_short'], 
                       c=all_results['hit_rate'], cmap='RdYlGn', alpha=0.6, s=50)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 1].set_xlabel('Sharpe Long')
    axes[0, 1].set_ylabel('Sharpe Short')
    axes[0, 1].set_title('Sharpe Ratio: Long vs Short')
    cbar = plt.colorbar(axes[0, 1].collections[0], ax=axes[0, 1])
    cbar.set_label('Hit Rate')
    
    # 3. Win Rate Long vs Short
    axes[1, 0].scatter(all_results['win_rate_long'], all_results['win_rate_short'],
                       c=all_results['hit_rate'], cmap='RdYlGn', alpha=0.6, s=50)
    axes[1, 0].axhline(y=0.5, color='red', linestyle='--')
    axes[1, 0].axvline(x=0.5, color='red', linestyle='--')
    axes[1, 0].set_xlabel('Win Rate Long')
    axes[1, 0].set_ylabel('Win Rate Short')
    axes[1, 0].set_title('Win Rate: Long vs Short')
    
    # 4. Profit Factor comparación
    top_pf = all_results.nlargest(20, 'profit_factor_long')
    x = range(len(top_pf))
    axes[1, 1].bar([i-0.2 for i in x], top_pf['profit_factor_long'], 
                   width=0.4, label='Long', color='green', alpha=0.7)
    axes[1, 1].bar([i+0.2 for i in x], top_pf['profit_factor_short'], 
                   width=0.4, label='Short', color='red', alpha=0.7)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels([f"{row['indicator'][:6]}" 
                                 for _, row in top_pf.iterrows()], 
                               rotation=45, ha='right', fontsize=8)
    axes[1, 1].axhline(y=1, color='black', linestyle='--', linewidth=0.5)
    axes[1, 1].set_ylabel('Profit Factor')
    axes[1, 1].set_title('Top 20 por Profit Factor Long')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    return all_results

print("✅ Función de comparación de indicadores lista")

✅ Función de comparación de indicadores lista


### 🎯 Análisis Mejorado de Persistencia de Indicadores

La función `momentum_persistence_analysis()` ahora puede analizar **todos los indicadores** implementados, no solo el momentum tradicional.

#### Opciones de análisis:

```python
# Analizar TODOS los indicadores (36 indicadores)
results = momentum_persistence_analysis(df, indicators='all')

# Analizar solo osciladores (RSI, Stochastic, CCI, Williams %R, etc.)
results = momentum_persistence_analysis(df, indicators='oscillators')

# Analizar indicadores de tendencia (MACD, ADX, Slope, Trix, etc.)
results = momentum_persistence_analysis(df, indicators='trend')

# Analizar indicadores específicos
results = momentum_persistence_analysis(df, indicators=['rsi', 'macd', 'adx'])

# Momentum tradicional (comportamiento original)
results = momentum_persistence_analysis(df, indicators='traditional')
```

#### Métricas calculadas por indicador:

- **Hit Rate**: % de veces que la señal coincide con dirección futura
- **Win Rate Long/Short**: % de operaciones ganadoras
- **Sharpe Long/Short**: Ratio retorno/riesgo direccional
- **Profit Factor**: Ganancia bruta / Pérdida bruta
- **Avg Return**: Retorno promedio por tipo de señal
- **N° Señales**: Cantidad de señales long/short/neutral

#### Función de comparación:

`compare_indicators_performance()` genera un **ranking completo** con:
- Top 20 indicadores por hit rate
- Rendimiento promedio por categoría
- Visualizaciones comparativas (scatter plots, barras)
- Análisis de Sharpe, Win Rate y Profit Factor

### 💡 Ejemplo de Uso del Análisis de Indicadores

Después de ejecutar `run_momentum_eda()`, puedes analizar el rendimiento de todos los indicadores:

In [8]:
# Ejemplo: Análisis completo de indicadores después de ejecutar run_momentum_eda()

# Si ya ejecutaste: data_btc = run_momentum_eda('BTC/USDT', start_date='2020-01-01')
# Puedes analizar el rendimiento de todos los indicadores en cada timeframe:

# Para timeframe diario:
# ranking_daily = compare_indicators_performance(data_btc['1d'], top_n=20)

# Para timeframe 4h:
# ranking_4h = compare_indicators_performance(data_btc['4h'], top_n=20)

# Para timeframe 1h:
# ranking_1h = compare_indicators_performance(data_btc['1h'], top_n=20)

# Ver resultados detallados de un grupo específico:
# oscillator_results = momentum_persistence_analysis(data_btc['1d'], indicators='oscillators')
# print(oscillator_results.head(10))

# Comparar categorías específicas:
# trend_indicators = momentum_persistence_analysis(data_btc['1d'], indicators='trend')
# print(f"\nMejor indicador de tendencia: {trend_indicators.iloc[0]['indicator'].upper()}")
# print(f"Hit Rate: {trend_indicators.iloc[0]['hit_rate']:.2%}")

# Exportar ranking completo a CSV:
# ranking_daily.to_csv('../data/stats/indicator_ranking_btcusdt_1d.csv', index=False)
# print("✓ Ranking exportado")

## 6️⃣ Fuerza Direccional y Continuidad

In [9]:
def directional_strength(df, window=20):
    """
    Mide la fuerza y consistencia de movimientos direccionales
    """
    df = df.copy()
    
    # Porcentaje de velas alcistas/bajistas en ventana (basado en candle_rtn)
    df['up_candles'] = (df['candle_rtn'] > 0).astype(int)
    df['pct_up'] = df['up_candles'].rolling(window).mean()
    
    # Rachas consecutivas (runs) - basadas en candle_rtn
    df['direction'] = np.where(df['candle_rtn'] > 0, 1, -1)
    df['run'] = (df['direction'] != df['direction'].shift()).cumsum()
    run_lengths = df.groupby('run')['direction'].agg(['count', 'first'])
    
    # Estadísticas de rachas
    avg_up_run = run_lengths[run_lengths['first'] == 1]['count'].mean()
    avg_down_run = run_lengths[run_lengths['first'] == -1]['count'].mean()
    max_up_run = run_lengths[run_lengths['first'] == 1]['count'].max()
    max_down_run = run_lengths[run_lengths['first'] == -1]['count'].max()
    
    return {
        'avg_up_run': avg_up_run,
        'avg_down_run': avg_down_run,
        'max_up_run': max_up_run,
        'max_down_run': max_down_run,
        'df': df
    }

print("✅ Función de fuerza direccional lista")

✅ Función de fuerza direccional lista


## 7️⃣ Visualización Completa

In [10]:
def plot_momentum_analysis(data_dict, ticker):
    """
    Genera visualizaciones completas del análisis
    """
    import os
    
    # Crear directorio si no existe
    img_dir = '../data/img'
    os.makedirs(img_dir, exist_ok=True)
    
    # Obtener rango de fechas del análisis
    date_start = None
    date_end = None
    for tf in ['1d', '4h', '1h']:
        if tf in data_dict:
            df = data_dict[tf]
            date_start = df.index.min().strftime('%Y%m%d')
            date_end = df.index.max().strftime('%Y%m%d')
            break
    
    # Fallback si no hay datos
    if not date_start or not date_end:
        from datetime import datetime
        date_start = datetime.now().strftime('%Y%m%d')
        date_end = date_start
    
    # Nombre limpio del ticker
    ticker_clean = ticker.replace('/', '').replace('-', '').lower()
    
    n_timeframes = len(data_dict)
    
    # === FIGURA 1: AUTOCORRELACIÓN ===
    fig1, axes = plt.subplots(n_timeframes, 3, figsize=(18, 5*n_timeframes))
    if n_timeframes == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (tf, df) in enumerate(data_dict.items()):
        returns = df['returns'].dropna()
        acorr_results = analyze_autocorrelation(returns, max_lags=40)
        
        # ACF de retornos
        axes[idx, 0].stem(range(len(acorr_results['acf'])), acorr_results['acf'], basefmt=' ')
        axes[idx, 0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        axes[idx, 0].axhline(y=1.96/np.sqrt(len(returns)), color='r', linestyle='--', linewidth=1)
        axes[idx, 0].axhline(y=-1.96/np.sqrt(len(returns)), color='r', linestyle='--', linewidth=1)
        axes[idx, 0].set_title(f'{tf.upper()} - ACF Retornos')
        axes[idx, 0].set_xlabel('Lag')
        axes[idx, 0].set_ylabel('Autocorrelación')
        
        # ACF de retornos absolutos (clustering de volatilidad)
        axes[idx, 1].stem(range(len(acorr_results['acf_abs'])), acorr_results['acf_abs'], basefmt=' ')
        axes[idx, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        axes[idx, 1].axhline(y=1.96/np.sqrt(len(returns)), color='r', linestyle='--', linewidth=1)
        axes[idx, 1].set_title(f'{tf.upper()} - ACF |Retornos| (Vol Clustering)')
        axes[idx, 1].set_xlabel('Lag')
        
        # Ljung-Box p-values
        axes[idx, 2].bar(range(len(acorr_results['ljung_box'])), 
                         acorr_results['ljung_box']['lb_pvalue'])
        axes[idx, 2].axhline(y=0.05, color='r', linestyle='--', label='α=0.05')
        axes[idx, 2].set_title(f'{tf.upper()} - Ljung-Box Test (p-values)')
        axes[idx, 2].set_xlabel('Lag')
        axes[idx, 2].set_ylabel('p-value')
        axes[idx, 2].legend()
    
    plt.tight_layout()
    autocorr_filename = f'momentum_autocorrelation_{ticker_clean}_{date_start}_{date_end}.png'
    autocorr_filepath = os.path.join(img_dir, autocorr_filename)
    plt.savefig(autocorr_filepath, dpi=150, bbox_inches='tight')
    print(f"✓ Gráfico guardado: {autocorr_filepath}")
    plt.show()
    
    # === FIGURA 2: PERSISTENCIA DE MOMENTUM ===
    fig2, axes = plt.subplots(1, n_timeframes, figsize=(6*n_timeframes, 5))
    if n_timeframes == 1:
        axes = [axes]
    
    for idx, (tf, df) in enumerate(data_dict.items()):
        # Usar indicators='traditional' para obtener el formato antiguo con períodos numéricos
        persistence = momentum_persistence_analysis(df, indicators='traditional')
        
        if not persistence.empty and 'period' in persistence.columns:
            # Convertir 'period' a numérico para graficar
            persistence_numeric = persistence[persistence['period'].apply(lambda x: str(x).isdigit())].copy()
            if not persistence_numeric.empty:
                persistence_numeric['period'] = persistence_numeric['period'].astype(int)
                persistence_numeric = persistence_numeric.sort_values('period')
                
                axes[idx].plot(persistence_numeric['period'], persistence_numeric['hit_rate'], 
                              marker='o', linewidth=2, markersize=8, label='Hit Rate')
                axes[idx].axhline(y=0.5, color='gray', linestyle='--', label='Random (50%)')
                axes[idx].set_title(f'{tf.upper()} - Momentum Persistence')
                axes[idx].set_xlabel('Lookback Period')
                axes[idx].set_ylabel('Hit Rate')
                axes[idx].legend()
                axes[idx].grid(True, alpha=0.3)
                axes[idx].set_ylim([0.4, 0.6])
    
    plt.tight_layout()
    persistence_filename = f'momentum_persistence_{ticker_clean}_{date_start}_{date_end}.png'
    persistence_filepath = os.path.join(img_dir, persistence_filename)
    plt.savefig(persistence_filepath, dpi=150, bbox_inches='tight')
    print(f"✓ Gráfico guardado: {persistence_filepath}")
    plt.show()
    
    # === FIGURA 3: DISTRIBUCIÓN DE RETORNOS Y QQ-PLOT ===
    fig3, axes = plt.subplots(n_timeframes, 2, figsize=(12, 5*n_timeframes))
    if n_timeframes == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (tf, df) in enumerate(data_dict.items()):
        returns = df['returns'].dropna()
        
        # Histograma con distribución normal
        axes[idx, 0].hist(returns, bins=100, density=True, alpha=0.7, color='blue')
        mu, std = returns.mean(), returns.std()
        x = np.linspace(returns.min(), returns.max(), 100)
        axes[idx, 0].plot(x, stats.norm.pdf(x, mu, std), 'r-', linewidth=2, label='Normal')
        axes[idx, 0].set_title(f'{tf.upper()} - Distribución Retornos')
        axes[idx, 0].set_xlabel('Retorno')
        axes[idx, 0].legend()
        
        # QQ-plot
        stats.probplot(returns, dist="norm", plot=axes[idx, 1])
        axes[idx, 1].set_title(f'{tf.upper()} - Q-Q Plot')
    
    plt.tight_layout()
    distribution_filename = f'returns_distribution_{ticker_clean}_{date_start}_{date_end}.png'
    distribution_filepath = os.path.join(img_dir, distribution_filename)
    plt.savefig(distribution_filepath, dpi=150, bbox_inches='tight')
    print(f"✓ Gráfico guardado: {distribution_filepath}")
    plt.show()

print("✅ Función de visualización lista")

✅ Función de visualización lista


## 8️⃣ Reporte Estadístico

In [11]:
def generate_statistical_report(data_dict, ticker):
    """
    Genera reporte estadístico completo
    """
    print("\n" + "="*80)
    print(f"📈 REPORTE DE ANÁLISIS DE MOMENTUM - {ticker}")
    print("="*80)
    
    for tf, df in data_dict.items():
        print(f"\n{'─'*80}")
        print(f"⏱️  TIMEFRAME: {tf.upper()}")
        print(f"{'─'*80}")
        
        returns = df['returns'].dropna()
        
        # Estadísticas descriptivas básicas
        print("\n📊 Estadísticas Descriptivas:")
        print(f"  • N observaciones: {len(returns):,}")
        print(f"  • Retorno medio: {returns.mean():.6f} ({returns.mean()*100:.4f}%)")
        print(f"  • Mediana: {returns.median():.6f}")
        print(f"  • Std Dev: {returns.std():.6f}")
        print(f"  • Sharpe Ratio: {returns.mean()/returns.std():.4f}")
        print(f"  • Skewness: {returns.skew():.4f}")
        print(f"  • Kurtosis: {returns.kurtosis():.4f}")
        
        # Test de normalidad
        _, p_value = stats.jarque_bera(returns)
        print(f"  • Jarque-Bera p-value: {p_value:.6f} {'✗ No normal' if p_value < 0.05 else '✓ Normal'}")
        
        # Autocorrelación
        acorr_results = analyze_autocorrelation(returns, max_lags=20)
        print(f"\n🔄 Autocorrelación:")
        print(f"  • ACF Lag-1: {acorr_results['acf'][1]:.6f}")
        print(f"  • ACF Lag-5: {acorr_results['acf'][5]:.6f}")
        print(f"  • ACF Lag-10: {acorr_results['acf'][10]:.6f}")
        
        sig_lags = np.where(np.abs(acorr_results['acf'][1:]) > 1.96/np.sqrt(len(returns)))[0]
        print(f"  • Lags significativos (α=0.05): {len(sig_lags)}/20")
        
        # Ljung-Box
        lb_significant = (acorr_results['ljung_box']['lb_pvalue'] < 0.05).sum()
        print(f"  • Ljung-Box rechaza H0 en: {lb_significant}/{len(acorr_results['ljung_box'])} lags")
        
        # Clustering de volatilidad
        print(f"\n💥 Clustering de Volatilidad:")
        print(f"  • ACF |returns| Lag-1: {acorr_results['acf_abs'][1]:.6f}")
        print(f"  • ACF returns² Lag-1: {acorr_results['acf_squared'][1]:.6f}")
        
        # Persistencia de momentum
        print(f"\n🎯 Persistencia de Momentum (Top 10 Indicadores):")
        persistence = momentum_persistence_analysis(df, indicators='all')
        if not persistence.empty:
            # Mostrar top 10 por hit rate
            top_10 = persistence.nlargest(10, 'hit_rate')
            for idx, row in top_10.iterrows():
                period_str = f"_{row['period']}" if row['period'] != 'default' else ""
                print(f"  • {row['indicator'].upper()}{period_str}: "
                      f"Hit={row['hit_rate']:.3f}, "
                      f"Sharpe L={row['sharpe_long']:.3f}, "
                      f"Sharpe S={row['sharpe_short']:.3f}, "
                      f"Win L={row['win_rate_long']:.2%}")
        
        # Fuerza direccional
        dir_stats = directional_strength(df)
        print(f"\n🔥 Fuerza Direccional:")
        print(f"  • Racha alcista promedio: {dir_stats['avg_up_run']:.2f} períodos")
        print(f"  • Racha bajista promedio: {dir_stats['avg_down_run']:.2f} períodos")
        print(f"  • Racha alcista máxima: {dir_stats['max_up_run']:.0f} períodos")
        print(f"  • Racha bajista máxima: {dir_stats['max_down_run']:.0f} períodos")
    
    print("\n" + "="*80 + "\n")

print("✅ Función de reporte estadístico lista")

✅ Función de reporte estadístico lista


## 9️⃣ Función Principal de Ejecución

In [12]:
def save_statistics_to_file(data_dict, ticker):
    """
    Guarda las estadísticas en archivos JSON y CSV para análisis comparativo
    """
    import json
    from datetime import datetime
    
    # Nombre limpio del ticker para archivos
    ticker_clean = ticker.replace('/', '').replace('-', '').lower()
    
    # Obtener rango de fechas del análisis (usar el timeframe más largo - diario)
    date_start = None
    date_end = None
    for tf in ['1d', '4h', '1h']:  # Prioridad: 1d > 4h > 1h
        if tf in data_dict:
            df = data_dict[tf]
            date_start = df.index.min().strftime('%Y%m%d')
            date_end = df.index.max().strftime('%Y%m%d')
            break
    
    # Fallback al timestamp si no hay datos
    if not date_start or not date_end:
        date_start = datetime.now().strftime('%Y%m%d')
        date_end = date_start
    
    # Diccionario para almacenar todas las estadísticas
    all_stats = {
        'ticker': ticker,
        'date_range': {
            'start': date_start,
            'end': date_end
        },
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'timeframes': {}
    }
    
    # Lista para CSV comparativo
    csv_data = []
    
    for tf, df in data_dict.items():
        returns = df['returns'].dropna()
        
        # Calcular todas las métricas
        acorr_results = analyze_autocorrelation(returns, max_lags=20)
        # Usar indicators='traditional' para obtener formato con períodos numéricos
        persistence = momentum_persistence_analysis(df, lookback_periods=[5, 10, 20, 30], indicators='traditional')
        dir_stats = directional_strength(df)
        
        # Construir diccionario de estadísticas para este timeframe
        tf_stats = {
            'descriptive': {
                'n_obs': int(len(returns)),
                'mean_return': float(returns.mean()),
                'median_return': float(returns.median()),
                'std_dev': float(returns.std()),
                'sharpe_ratio': float(returns.mean() / returns.std()),
                'skewness': float(returns.skew()),
                'kurtosis': float(returns.kurtosis()),
                'min_return': float(returns.min()),
                'max_return': float(returns.max())
            },
            'autocorrelation': {
                'acf_lag1': float(acorr_results['acf'][1]),
                'acf_lag5': float(acorr_results['acf'][5]),
                'acf_lag10': float(acorr_results['acf'][10]),
                'significant_lags': int(np.sum(np.abs(acorr_results['acf'][1:21]) > 1.96/np.sqrt(len(returns)))),
                'ljung_box_rejections': int((acorr_results['ljung_box']['lb_pvalue'] < 0.05).sum())
            },
            'volatility_clustering': {
                'acf_abs_lag1': float(acorr_results['acf_abs'][1]),
                'acf_squared_lag1': float(acorr_results['acf_squared'][1])
            },
            'momentum_persistence': {},
            'directional_strength': {
                'avg_up_run': float(dir_stats['avg_up_run']),
                'avg_down_run': float(dir_stats['avg_down_run']),
                'max_up_run': int(dir_stats['max_up_run']),
                'max_down_run': int(dir_stats['max_down_run'])
            }
        }
        
        # Agregar persistencia de momentum
        if not persistence.empty:
            for _, row in persistence.iterrows():
                # Convertir period a int si es numérico
                try:
                    lookback = int(row['period'])
                except (ValueError, TypeError):
                    continue  # Saltar si no es numérico
                
                tf_stats['momentum_persistence'][f'lookback_{lookback}'] = {
                    'hit_rate': float(row['hit_rate']),
                    'avg_return_long': float(row['avg_return_long']),
                    'avg_return_short': float(row['avg_return_short']),
                    'sharpe_long': float(row['sharpe_long']),
                    'sharpe_short': float(row['sharpe_short']),
                    'n_obs': int(row['n_obs'])
                }
                
                # Agregar a CSV data
                csv_data.append({
                    'ticker': ticker,
                    'timeframe': tf,
                    'lookback': lookback,
                    'hit_rate': row['hit_rate'],
                    'sharpe_long': row['sharpe_long'],
                    'sharpe_short': row['sharpe_short'],
                    'mean_return': returns.mean(),
                    'volatility': returns.std(),
                    'sharpe_ratio': returns.mean() / returns.std(),
                    'acf_lag1': acorr_results['acf'][1],
                    'vol_clustering': acorr_results['acf_abs'][1]
                })
        
        all_stats['timeframes'][tf] = tf_stats
    
    # Crear directorio si no existe
    import os
    stats_dir = '../data/stats'
    os.makedirs(stats_dir, exist_ok=True)
    
    # Guardar JSON completo
    json_filename = f'momentum_stats_{ticker_clean}_{date_start}_{date_end}.json'
    json_filepath = os.path.join(stats_dir, json_filename)
    with open(json_filepath, 'w', encoding='utf-8') as f:
        json.dump(all_stats, f, indent=2, ensure_ascii=False)
    print(f"✓ Estadísticas guardadas: {json_filepath}")
    
    # Guardar CSV para comparación rápida
    if csv_data:
        csv_filename = f'momentum_summary_{ticker_clean}_{date_start}_{date_end}.csv'
        csv_filepath = os.path.join(stats_dir, csv_filename)
        df_csv = pd.DataFrame(csv_data)
        df_csv.to_csv(csv_filepath, index=False)
        print(f"✓ Resumen CSV guardado: {csv_filepath}")
    
    return all_stats


def run_momentum_eda(ticker='BTC/USDT', start_date='2020-01-01', save_stats=True, use_cached=True):
    """
    Ejecuta el análisis exploratorio completo
    
    Parameters:
    -----------
    ticker : str
        Par de trading en formato 'BTC/USDT', 'ETH/USDT', etc.
    start_date : str
        Fecha de inicio en formato 'YYYY-MM-DD'
    save_stats : bool
        Si True, guarda las estadísticas en archivos JSON y CSV
    use_cached : bool
        Si True, intenta cargar datos guardados antes de descargar
    """
    print("🚀 Iniciando Análisis Exploratorio de Momentum\n")
    
    # Intentar cargar datos guardados si use_cached=True
    data_raw = None
    if use_cached:
        print("🔍 Buscando datos guardados en caché...")
        data_raw = load_saved_data(ticker)
        
        if data_raw:
            print("✓ Usando datos guardados\n")
        else:
            print("⚠️  No se encontraron datos en caché, descargando...\n")
    
    # Descargar datos si no hay caché o use_cached=False
    if data_raw is None:
        data_raw = download_multi_timeframe_data(ticker, start_date, save_data=True)
    
    if not data_raw:
        print("❌ Error: No se pudieron obtener datos")
        return None
    
    # Calcular retornos y momentum
    data_processed = {}
    for tf, df in data_raw.items():
        print(f"⚙️  Procesando {tf}...")
        data_processed[tf] = calculate_returns_and_momentum(df)
    
    # Generar visualizaciones
    print("\n📊 Generando visualizaciones...")
    plot_momentum_analysis(data_processed, ticker)
    
    # Generar reporte
    generate_statistical_report(data_processed, ticker)
    
    # Guardar estadísticas
    if save_stats:
        print("\n💾 Guardando estadísticas para análisis comparativo...")
        stats = save_statistics_to_file(data_processed, ticker)
    
    return data_processed

print("✅ Función principal lista para ejecutar con caché Parquet")

✅ Función principal lista para ejecutar con caché Parquet


---
## 🚀 EJECUCIÓN DEL ANÁLISIS

Ejecuta las siguientes celdas para realizar el análisis completo de momentum para Bitcoin (BTC/USDT).

**Nota:** Los datos ahora provienen de Binance Spot vía CCXT, con datos completos en 1h, 4h y diarios.

### Análisis adicional para Bitcoin (BTC)

In [ ]:
# Ejecutar análisis completo para BTC
data_btc = run_momentum_eda('BTC/USDT', start_date='2017-08-17')

### Análisis adicional para Ethereum (ETH)

Si deseas comparar con ETH, ejecuta la siguiente celda:

In [ ]:
# Ejecutar análisis completo para ETH (opcional)
data_eth = run_momentum_eda('ETH/USDT', start_date='2017-01-01')

### Análisis adicional para Litecoin (LTC)

In [ ]:
data_eth = run_momentum_eda('LTC/USDT', start_date='2017-01-01')

### Análisis adicional para Zcash (ZEC)

In [ ]:
data_eth = run_momentum_eda('ZEC/USDT', start_date='2017-01-01')

### Análisis adicional para Solana (SOL)

In [ ]:
data_eth = run_momentum_eda('SOL/USDT', start_date='2017-01-01')

### Análisis adicional para Uniswap (UNI)

In [ ]:
data_eth = run_momentum_eda('UNI/USDT', start_date='2017-01-01')

---
## 📊 Interpretación de Resultados

### ✅ Indicadores de Momentum Positivo:
- **Hit Rate > 50%**: El momentum pasado predice correctamente la dirección futura
- **ACF significativo**: Autocorrelación explotable en retornos
- **ACF |returns| alto**: Clustering de volatilidad (útil para dimensionar posiciones)
- **Rachas largas**: Tendencias fuertes y sostenidas

### 📈 Métricas Clave:
1. **Autocorrelación (ACF)**: Mide dependencia temporal de retornos
2. **Ljung-Box Test**: Confirma presencia de autocorrelación (p-value < 0.05)
3. **Hit Rate**: Probabilidad de que señal de momentum acierte dirección
4. **Sharpe Direccional**: Retorno/riesgo de estrategias long/short
5. **Clustering de Volatilidad**: Persistencia de períodos de alta/baja volatilidad

### 🎯 Aplicación a Trading:
- **Timeframes con Hit Rate > 52%**: Candidatos para estrategias momentum
- **ACF positivo en lag-1**: Señal de continuación de tendencias cortas
- **Clustering de volatilidad alto**: Implementar gestión dinámica de posiciones

---
## 🔄 ANÁLISIS COMPARATIVO ENTRE ACTIVOS

Las siguientes funciones permiten comparar estadísticas entre múltiples activos (BTC, ETH, etc.).

---
## 📚 Guía de Interpretación de Resultados Comparativos

### 🎯 **Métricas Clave Explicadas**

#### 1. **Hit Rate** (Tasa de Acierto)
- **Qué mide**: Porcentaje de veces que la señal del indicador predice correctamente la dirección del siguiente movimiento
- **Interpretación**:
  - **> 52%**: Indicador con capacidad predictiva, útil para trading
  - **50%**: Aleatorio, sin capacidad predictiva
  - **< 48%**: Contrarian (considerar señales inversas)
- **Ejemplo**: Hit Rate 55% = 55 de cada 100 señales aciertan la dirección

#### 2. **Win Rate Long/Short** (Tasa de Victorias)
- **Qué mide**: Porcentaje de operaciones ganadoras por tipo de posición
- **Diferencia con Hit Rate**: 
  - Hit Rate: considera magnitud + dirección
  - Win Rate: solo cuenta operaciones ganadoras
- **Interpretación**:
  - Win Rate 60% Long = 60% de posiciones long son rentables
  - Útil para estrategias discretas (entrar/salir)

#### 3. **Sharpe Ratio Long/Short** (Retorno Ajustado por Riesgo)
- **Qué mide**: Retorno promedio dividido por volatilidad de retornos
- **Interpretación**:
  - **> 1.0**: Excelente relación retorno/riesgo
  - **0.5 - 1.0**: Bueno, estrategia viable
  - **< 0.5**: Malo, mucho riesgo para poco retorno
  - **Negativo**: Pérdidas sistemáticas
- **Ejemplo**: Sharpe 1.5 = por cada unidad de riesgo obtienes 1.5 unidades de retorno

#### 4. **Profit Factor** (Factor de Beneficio)
- **Qué mide**: Ganancia bruta total / Pérdida bruta total
- **Interpretación**:
  - **> 2.0**: Excelente, ganas el doble de lo que pierdes
  - **1.5 - 2.0**: Muy bueno, estrategia rentable
  - **1.0 - 1.5**: Viable pero marginal
  - **< 1.0**: Pérdidas netas, evitar
- **Ejemplo**: PF = 2.5 significa que por cada $100 perdidos, ganas $250

#### 5. **Avg Return Long/Short** (Retorno Promedio)
- **Qué mide**: Retorno porcentual promedio por operación
- **Interpretación**:
  - Positivo = rentable en promedio
  - Negativo = pérdidas en promedio
- **Importante**: Considerar junto con volatilidad (Sharpe)

---

### 📊 **Cómo Interpretar el Ranking de Indicadores**

#### **Top Indicadores por Hit Rate:**
```
1. RSI_20: Hit Rate=54.2%, Win Long=58%, Sharpe L=0.85
   → Buen indicador de reversión, especialmente para long
   → Sharpe decente, Win Rate alto
   → CONCLUSIÓN: Útil para comprar en sobreventa

2. MACD: Hit Rate=52.8%, Win Long=51%, Sharpe L=1.12
   → Hit rate moderado pero Sharpe excelente
   → Relación retorno/riesgo superior
   → CONCLUSIÓN: Mejor para gestión de riesgo que frecuencia
```

#### **Indicadores a Evitar:**
- Hit Rate < 48%: Sin capacidad predictiva o contrarian
- Sharpe negativo: Pérdidas consistentes
- Profit Factor < 1.0: Más pérdidas que ganancias

---

### 📈 **Análisis por Categoría**

#### **Osciladores (RSI, Stochastic, Williams %R)**
- **Función**: Identificar sobrecompra/sobreventa
- **Mejor en**: Mercados laterales o con reversión
- **Buscar**: Win Rate alto en zonas extremas

#### **Tendencia (MACD, ADX, Slope)**
- **Función**: Seguir tendencias establecidas
- **Mejor en**: Mercados trending
- **Buscar**: Sharpe Ratio alto, consistencia

#### **Volatilidad (Bollinger Bands, ATR, RVI)**
- **Función**: Medir expansión/contracción
- **Mejor en**: Breakouts y gestión de riesgo
- **Buscar**: Señales de compresión → expansión

---

### 🎲 **Ejemplos Prácticos de Interpretación**

#### **Escenario 1: BTC Timeframe 1H**
```
Top 3 Indicadores:
1. RSI_20: Hit=54%, Sharpe=0.92, PF=1.8
2. Williams_14: Hit=53%, Sharpe=0.78, PF=1.6
3. Stochastic: Hit=52%, Sharpe=1.05, PF=1.9

INTERPRETACIÓN:
→ Mercado favorable para REVERSIÓN A LA MEDIA
→ Indicadores de sobreventa/sobrecompra funcionan bien
→ Estrategia sugerida: Comprar RSI<30, vender RSI>70
→ Gestión de riesgo: Usar Stochastic (mejor Sharpe)
```

#### **Escenario 2: ETH Timeframe 4H**
```
Top 3 Indicadores:
1. MACD: Hit=56%, Sharpe=1.25, PF=2.1
2. ADX: Hit=55%, Sharpe=1.18, PF=1.9
3. Slope_20: Hit=54%, Sharpe=0.95, PF=1.7

INTERPRETACIÓN:
→ Mercado favorable para SEGUIMIENTO DE TENDENCIA
→ Indicadores direccionales dominan
→ Estrategia sugerida: Entrar con MACD cross, confirmar con ADX>25
→ Hold más tiempo, tendencias más fuertes
```

---

### 🔍 **Cómo Usar los Gráficos de Comparación**

#### **Gráfico 1: Hit Rate por Indicador**
- Busca barras que superen la línea roja (50%)
- Cuanto más larga la barra, mejor predicción

#### **Gráfico 2: Sharpe Long vs Short**
- Cuadrante superior derecho: Buenos en ambos lados
- Puntos verdes: Alto hit rate
- Puntos rojos: Bajo hit rate

#### **Gráfico 3: Win Rate Long vs Short**
- Diagonal = consistencia ambos lados
- Arriba derecha = buenos para estrategias long/short

#### **Gráfico 4: Profit Factor Comparación**
- Barras verdes (long) vs rojas (short)
- Busca barras que superen línea negra (PF=1.0)

---

### 💡 **Recomendaciones Estratégicas**

#### **Para Construir una Estrategia:**

1. **Seleccionar por Timeframe:**
   - 1H: Favor reversión (RSI, Williams)
   - 4H: Balance reversión/tendencia
   - 1D: Favor tendencia (MACD, ADX)

2. **Combinar Indicadores:**
   - Usar top 2-3 con alta correlación inversa
   - Ejemplo: RSI (reversión) + MACD (tendencia) = señales más robustas

3. **Validar con Métricas:**
   - Hit Rate mínimo: 52%
   - Sharpe mínimo: 0.5
   - Profit Factor mínimo: 1.3

4. **Backtesting:**
   - Los resultados son en SAMPLE (in-sample)
   - Validar en período diferente (out-of-sample)
   - Considerar costos de transacción

---

### ⚠️ **Advertencias Importantes**

1. **No usar solo Hit Rate**: Un indicador con 55% hit rate pero Sharpe negativo pierde dinero
2. **Contexto de mercado**: Indicadores funcionan diferente en trending vs lateral
3. **Overfitting**: Top indicador en un período puede fallar en otro
4. **Combinación es clave**: Múltiples indicadores mediocres pueden hacer una estrategia excelente
5. **Gestión de riesgo**: Incluso con 60% win rate, sin stop-loss pierdes todo en una operación

---

### 📝 **Checklist de Evaluación**

Para considerar un indicador viable:
- ✅ Hit Rate > 52%
- ✅ Win Rate > 50% en al menos un lado (long o short)
- ✅ Sharpe Ratio > 0.5
- ✅ Profit Factor > 1.3
- ✅ N° de señales suficiente (> 100 operaciones)
- ✅ Consistencia entre timeframes similares

Si un indicador cumple 4+ criterios, vale la pena incluirlo en tu estrategia.

In [ ]:
def compare_multiple_assets(tickers=['BTC/USDT', 'ETH/USDT'], start_date='2020-01-01'):
    """
    Analiza múltiples activos y genera una comparación consolidada
    
    Parameters:
    -----------
    tickers : list
        Lista de tickers en formato 'BTC/USDT', 'ETH/USDT', etc.
    start_date : str
        Fecha de inicio en formato 'YYYY-MM-DD'
    """
    print("🔄 Iniciando análisis comparativo de múltiples activos\n")
    
    all_results = {}
    
    # Analizar cada activo
    for ticker in tickers:
        print(f"\n{'='*80}")
        print(f"Analizando {ticker}...")
        print(f"{'='*80}\n")
        
        data = run_momentum_eda(ticker, start_date, save_stats=True)
        all_results[ticker] = data
    
    # Crear tabla comparativa consolidada
    print("\n\n" + "="*100)
    print("📊 TABLA COMPARATIVA - MÉTRICAS CLAVE")
    print("="*100)
    
    comparison_data = []
    
    for ticker in tickers:
        if ticker not in all_results or all_results[ticker] is None:
            continue
        
        for tf in ['1h', '4h', '1d']:
            if tf not in all_results[ticker]:
                continue
                
            df = all_results[ticker][tf]
            returns = df['returns'].dropna()
            acorr = analyze_autocorrelation(returns, max_lags=20)
            # Usar traditional para obtener período 30
            pers = momentum_persistence_analysis(df, lookback_periods=[30], indicators='traditional')
            
            if not pers.empty:
                comparison_data.append({
                    'Ticker': ticker,
                    'TF': tf.upper(),
                    'Retorno %': f"{returns.mean()*100:.3f}",
                    'Vol %': f"{returns.std()*100:.2f}",
                    'Sharpe': f"{returns.mean()/returns.std():.3f}",
                    'ACF-1': f"{acorr['acf'][1]:.3f}",
                    'Vol Clust': f"{acorr['acf_abs'][1]:.3f}",
                    'Hit Rate': f"{pers.iloc[0]['hit_rate']:.3f}",
                    'Sharpe L': f"{pers.iloc[0]['sharpe_long']:.3f}"
                })
    
    # Mostrar tabla
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Guardar comparación consolidada
    import os
    stats_dir = '../data/stats'
    os.makedirs(stats_dir, exist_ok=True)
    
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    comparison_filename = f'momentum_comparison_{timestamp}.csv'
    comparison_filepath = os.path.join(stats_dir, comparison_filename)
    df_comparison.to_csv(comparison_filepath, index=False)
    print(f"\n✓ Comparación guardada: {comparison_filepath}")
    print("="*100)
    
    return all_results, df_comparison


def load_and_compare_saved_stats(json_files):
    """
    Carga archivos JSON guardados previamente y genera comparación
    
    Parameters:
    -----------
    json_files : list
        Lista de rutas a archivos JSON de estadísticas
    """
    import json
    
    print("📂 Cargando estadísticas guardadas...\n")
    
    all_stats = []
    
    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                stats = json.load(f)
                all_stats.append(stats)
                print(f"✓ Cargado: {file_path} ({stats['ticker']})")
        except Exception as e:
            print(f"✗ Error al cargar {file_path}: {e}")
    
    if not all_stats:
        print("❌ No se pudieron cargar archivos")
        return None
    
    # Crear tabla comparativa
    print("\n" + "="*100)
    print("📊 COMPARACIÓN DE ACTIVOS (desde archivos guardados)")
    print("="*100 + "\n")
    
    comparison_rows = []
    
    for stat in all_stats:
        ticker = stat['ticker']
        for tf, tf_data in stat['timeframes'].items():
            desc = tf_data['descriptive']
            acorr = tf_data['autocorrelation']
            vol = tf_data['volatility_clustering']
            
            # Obtener hit rate de lookback 30
            hit_rate = None
            sharpe_long = None
            if 'lookback_30' in tf_data['momentum_persistence']:
                hit_rate = tf_data['momentum_persistence']['lookback_30']['hit_rate']
                sharpe_long = tf_data['momentum_persistence']['lookback_30']['sharpe_long']
            
            comparison_rows.append({
                'Ticker': ticker,
                'TF': tf.upper(),
                'Retorno %': f"{desc['mean_return']*100:.3f}",
                'Vol %': f"{desc['std_dev']*100:.2f}",
                'Sharpe': f"{desc['sharpe_ratio']:.3f}",
                'ACF-1': f"{acorr['acf_lag1']:.3f}",
                'Vol Clust': f"{vol['acf_abs_lag1']:.3f}",
                'Hit Rate': f"{hit_rate:.3f}" if hit_rate else "N/A",
                'Sharpe L': f"{sharpe_long:.3f}" if sharpe_long else "N/A"
            })
    
    df_comp = pd.DataFrame(comparison_rows)
    print(df_comp.to_string(index=False))
    print("\n" + "="*100)
    
    return df_comp

print("✅ Funciones de comparación listas")

### Ejemplo 1: Comparar BTC y ETH en una sola ejecución

Esta función ejecuta el análisis completo para múltiples activos y genera una tabla comparativa:

In [ ]:
# Comparar BTC y ETH (y otros activos si deseas)
# results, comparison_table = compare_multiple_assets(
#     tickers=['BTC/USDT', 'ETH/USDT'],  # Puedes agregar más: 'SOL/USDT', 'BNB/USDT', etc.
#     start_date='2020-01-01'
# )

### Ejemplo 2: Cargar archivos JSON guardados previamente

Si ya ejecutaste los análisis en diferentes momentos, puedes cargar los archivos JSON:

In [ ]:
import os
import glob

# Obtener todos los archivos JSON en ../data/stats
stats_dir = '../data/stats'
json_files = glob.glob(os.path.join(stats_dir, '*.json'))

# Mostrar archivos encontrados
print(f"📂 Archivos JSON encontrados en {stats_dir}:")
for file in json_files:
    print(f"  • {os.path.basename(file)}")
print(f"\nTotal: {len(json_files)} archivos\n")

# Ejemplo de cómo cargar y comparar archivos guardados
comparison_df = load_and_compare_saved_stats(json_files)

---
## 📁 Archivos Generados

Cada vez que ejecutas `run_momentum_eda()`, se generan los siguientes archivos:

### **1. Archivos de Visualización (PNG)**
- `momentum_autocorrelation.png` - Gráficos de ACF, clustering de volatilidad y Ljung-Box
- `momentum_persistence.png` - Hit rates por lookback period
- `returns_distribution.png` - Histogramas y Q-Q plots

### **2. Archivos de Estadísticas (JSON)**
- `momentum_stats_{TICKER}_{FECHA_INICIO}_{FECHA_FIN}.json` - Datos completos en formato JSON
  - Todas las métricas por timeframe
  - Fácil de procesar programáticamente
  - Incluye metadata (ticker, rango de fechas)
  - **Ejemplo**: `momentum_stats_BTC_USDT_20200101_20251110.json`

### **3. Archivos de Resumen (CSV)**
- `momentum_summary_{TICKER}_{FECHA_INICIO}_{FECHA_FIN}.csv` - Tabla resumen para Excel/análisis rápido
  - Una fila por combinación ticker-timeframe-lookback
  - Métricas clave: hit_rate, sharpe, volatilidad, ACF
  - **Ejemplo**: `momentum_summary_BTC_USDT_20200101_20251110.csv`

### **4. Archivo de Comparación (CSV)**
- `momentum_comparison_{TIMESTAMP}.csv` - Comparación entre múltiples activos
  - Generado por `compare_multiple_assets()`
  - Tabla consolidada lado a lado

### **Ejemplo de uso:**
```python
# Ejecutar análisis individual (genera archivos automáticamente)
data_btc = run_momentum_eda('BTC/USDT', start_date='2020-01-01')
# Genera: momentum_stats_BTC_USDT_20200101_20251110.json

# Comparar múltiples activos
results, comparison = compare_multiple_assets(
    tickers=['BTC/USDT', 'ETH/USDT', 'SOL/USDT'],
    start_date='2020-01-01'
)

# Cargar análisis previos
comparison_df = load_and_compare_saved_stats([
    'momentum_stats_BTC_USDT_20200101_20251110.json',
    'momentum_stats_ETH_USDT_20200101_20251110.json'
])
```

---
## 🔬 Grid Search de Estrategias con MLflow

Sistema de experimentación automatizada para optimizar parámetros de estrategias de trading usando Grid Search y tracking con MLflow.

In [13]:
import mlflow
import mlflow.sklearn
from itertools import product
import json
from typing import Dict, List, Any, Tuple
import time

# Configurar MLflow
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("momentum_trading_strategies")

print("✅ MLflow configurado")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Experimento: momentum_trading_strategies")

✅ MLflow configurado
   Tracking URI: file:../mlruns
   Experimento: momentum_trading_strategies


In [ ]:
def calculate_indicator_and_signals(df, indicator, params):
    """
    Calcula dinámicamente el indicador y sus señales según parámetros
    
    Parameters:
    -----------
    df : DataFrame
        Datos OHLCV básicos
    indicator : str
        Nombre del indicador
    params : dict
        Parámetros específicos (period, thresholds, etc.)
    
    Returns:
    --------
    DataFrame con columna 'signal' añadida
    """
    import pandas_ta as ta
    
    df = df.copy()
    
    # Extraer parámetros comunes
    period = params.get('period', 14)
    
    # === OSCILLATORS - REVERSAL ===
    if indicator == 'rsi':
        overbought = params.get('overbought', 70)
        oversold = params.get('oversold', 30)
        
        indicator_values = ta.rsi(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > overbought, 'signal'] = -1  # Señal bajista
        df.loc[indicator_values < oversold, 'signal'] = 1     # Señal alcista
    
    elif indicator == 'willr':
        high_threshold = params.get('high_threshold', -20)
        low_threshold = params.get('low_threshold', -80)
        
        indicator_values = ta.willr(df['High'], df['Low'], df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > high_threshold, 'signal'] = -1
        df.loc[indicator_values < low_threshold, 'signal'] = 1
    
    elif indicator == 'stoch':
        k_period = params.get('k', 14)
        d_period = params.get('d', 3)
        overbought = params.get('overbought', 80)
        oversold = params.get('oversold', 20)
        
        stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=k_period, d=d_period)
        k_values = stoch[f'STOCHk_{k_period}_{d_period}_3']
        
        df['signal'] = 0
        df.loc[k_values > overbought, 'signal'] = -1
        df.loc[k_values < oversold, 'signal'] = 1
    
    elif indicator == 'cci':
        overbought = params.get('overbought', 100)
        oversold = params.get('oversold', -100)
        
        indicator_values = ta.cci(df['High'], df['Low'], df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > overbought, 'signal'] = -1
        df.loc[indicator_values < oversold, 'signal'] = 1
    
    elif indicator == 'rsx':
        overbought = params.get('overbought', 70)
        oversold = params.get('oversold', 30)
        
        indicator_values = ta.rsx(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > overbought, 'signal'] = -1
        df.loc[indicator_values < oversold, 'signal'] = 1
    
    elif indicator == 'cmo':
        overbought = params.get('overbought', 50)
        oversold = params.get('oversold', -50)
        
        indicator_values = ta.cmo(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > overbought, 'signal'] = -1
        df.loc[indicator_values < oversold, 'signal'] = 1
    
    elif indicator == 'uo':
        overbought = params.get('overbought', 70)
        oversold = params.get('oversold', 30)
        
        indicator_values = ta.uo(df['High'], df['Low'], df['Close'])
        df['signal'] = 0
        df.loc[indicator_values > overbought, 'signal'] = -1
        df.loc[indicator_values < oversold, 'signal'] = 1
    
    # === TREND INDICATORS ===
    elif indicator == 'macd':
        fast = params.get('fast', 12)
        slow = params.get('slow', 26)
        signal_period = params.get('signal', 9)
        
        macd = ta.macd(df['Close'], fast=fast, slow=slow, signal=signal_period)
        macd_line = macd[f'MACD_{fast}_{slow}_{signal_period}']
        signal_line = macd[f'MACDs_{fast}_{slow}_{signal_period}']
        
        df['signal'] = 0
        df.loc[macd_line > signal_line, 'signal'] = 1   # Cruce alcista
        df.loc[macd_line < signal_line, 'signal'] = -1  # Cruce bajista
    
    elif indicator == 'adx':
        threshold = params.get('threshold', 25)
        
        adx_data = ta.adx(df['High'], df['Low'], df['Close'], length=period)
        adx_values = adx_data[f'ADX_{period}']
        dmp = adx_data[f'DMP_{period}']
        dmn = adx_data[f'DMN_{period}']
        
        df['signal'] = 0
        df.loc[(adx_values > threshold) & (dmp > dmn), 'signal'] = 1
        df.loc[(adx_values > threshold) & (dmn > dmp), 'signal'] = -1
    
    elif indicator == 'er':
        threshold = params.get('threshold', 0.3)
        
        indicator_values = ta.er(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > threshold, 'signal'] = 1
        df.loc[indicator_values <= threshold, 'signal'] = -1
    
    elif indicator == 'slope':
        threshold = params.get('threshold', 0)
        
        indicator_values = ta.slope(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > threshold, 'signal'] = 1
        df.loc[indicator_values < threshold, 'signal'] = -1
    
    elif indicator == 'trix':
        threshold = params.get('threshold', 0)
        
        indicator_values = ta.trix(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > threshold, 'signal'] = 1
        df.loc[indicator_values < threshold, 'signal'] = -1
    
    elif indicator == 'ao':
        threshold = params.get('threshold', 0)
        
        indicator_values = ta.ao(df['High'], df['Low'])
        df['signal'] = 0
        df.loc[indicator_values > threshold, 'signal'] = 1
        df.loc[indicator_values < threshold, 'signal'] = -1
    
    # === MOMENTUM ADVANCED ===
    elif indicator == 'inertia':
        high_threshold = params.get('high_threshold', 60)
        low_threshold = params.get('low_threshold', 40)
        
        indicator_values = ta.inertia(df['Close'], length=period)
        df['signal'] = 0
        df.loc[indicator_values > high_threshold, 'signal'] = 1
        df.loc[indicator_values < low_threshold, 'signal'] = -1
    
    elif indicator == 'bop':
        threshold = params.get('threshold', 0)
        
        indicator_values = ta.bop(df['Open'], df['High'], df['Low'], df['Close'])
        df['signal'] = 0
        df.loc[indicator_values > threshold, 'signal'] = 1
        df.loc[indicator_values < threshold, 'signal'] = -1
    
    else:
        # Indicador no soportado
        df['signal'] = 0
    
    return df


def combine_indicator_signals(df, indicators_configs, combination_method='AND'):
    """
    Combina señales de múltiples indicadores usando lógica configurable
    
    Parameters:
    -----------
    df : DataFrame
        Datos OHLCV básicos
    indicators_configs : list
        Lista de configuraciones de indicadores a combinar
        Ejemplo:
        [
            {'indicator': 'rsi', 'params': {'period': 14, 'overbought': 70, 'oversold': 30}},
            {'indicator': 'macd', 'params': {'fast': 12, 'slow': 26, 'signal': 9}},
            {'indicator': 'adx', 'params': {'period': 14, 'threshold': 25}}
        ]
    combination_method : str
        Método de combinación:
        - 'AND': Todas las señales deben coincidir (alcista=1 o bajista=-1)
        - 'OR': Al menos una señal debe ser != 0
        - 'MAJORITY': Señal mayoritaria (al menos 50%+1)
        - 'WEIGHTED': Promedio ponderado (requiere 'weight' en cada config)
        - 'UNANIMOUS_LONG': Solo long si TODOS son alcistas (1), resto neutral
        - 'UNANIMOUS_SHORT': Solo short si TODOS son bajistas (-1), resto neutral
    
    Returns:
    --------
    DataFrame con columna 'signal' combinada
    """
    df = df.copy()
    
    # Calcular señales de cada indicador
    signals = []
    weights = []
    
    for config in indicators_configs:
        indicator = config['indicator']
        params = config.get('params', {})
        weight = config.get('weight', 1.0)
        
        # Calcular señal del indicador
        df_temp = calculate_indicator_and_signals(df, indicator, params)
        
        if 'signal' in df_temp.columns:
            signal_col_name = f"signal_{indicator}_{len(signals)}"
            df[signal_col_name] = df_temp['signal']
            signals.append(signal_col_name)
            weights.append(weight)
    
    if len(signals) == 0:
        df['signal'] = 0
        return df
    
    # Combinar señales según método
    if combination_method == 'AND':
        # Todas deben ser iguales y != 0
        df['signal'] = df[signals[0]]
        for sig in signals[1:]:
            # Si alguna es diferente o es 0, resultado es 0
            df['signal'] = np.where(
                (df['signal'] == df[sig]) & (df['signal'] != 0),
                df['signal'],
                0
            )
    
    elif combination_method == 'OR':
        # Al menos una señal != 0
        # Priorizar señales alcistas si hay conflicto
        signal_sum = sum([df[sig] for sig in signals])
        df['signal'] = np.sign(signal_sum)
    
    elif combination_method == 'MAJORITY':
        # Voto mayoritario
        signal_sum = sum([df[sig] for sig in signals])
        threshold = len(signals) / 2
        df['signal'] = np.where(signal_sum > threshold, 1,
                                np.where(signal_sum < -threshold, -1, 0))
    
    elif combination_method == 'WEIGHTED':
        # Promedio ponderado
        weighted_sum = sum([df[sig] * w for sig, w in zip(signals, weights)])
        total_weight = sum(weights)
        df['signal'] = np.sign(weighted_sum / total_weight)
    
    elif combination_method == 'UNANIMOUS_LONG':
        # Solo long si TODOS son 1
        all_long = pd.Series([True] * len(df), index=df.index)
        for sig in signals:
            all_long = all_long & (df[sig] == 1)
        df['signal'] = np.where(all_long, 1, 0)
    
    elif combination_method == 'UNANIMOUS_SHORT':
        # Solo short si TODOS son -1
        all_short = pd.Series([True] * len(df), index=df.index)
        for sig in signals:
            all_short = all_short & (df[sig] == -1)
        df['signal'] = np.where(all_short, -1, 0)
    
    else:
        # Default: OR
        signal_sum = sum([df[sig] for sig in signals])
        df['signal'] = np.sign(signal_sum)
    
    # Limpiar columnas temporales
    df = df.drop(columns=signals)
    
    return df


def backtest_strategy(df, indicator, params, position_type='long', indicators_combo=None, 
                      combination_method='AND'):
    """
    Backtesting completo: calcula indicador(es), señales y métricas
    Soporta indicadores individuales o combinaciones
    
    Parameters:
    -----------
    df : DataFrame
        Datos OHLCV básicos (open, high, low, close, volume, returns)
    indicator : str or None
        Nombre del indicador individual (ej: 'rsi', 'macd', 'stoch')
        Si indicators_combo es provisto, este parámetro se ignora
    params : dict
        Parámetros de la estrategia (period, thresholds, etc.)
        Solo usado si indicator es un string (indicador individual)
    position_type : str
        'long', 'short' o 'both'
    indicators_combo : list or None
        Lista de configuraciones para estrategia multi-indicador
        Ejemplo:
        [
            {'indicator': 'rsi', 'params': {'period': 14, 'overbought': 70, 'oversold': 30}},
            {'indicator': 'macd', 'params': {'fast': 12, 'slow': 26, 'signal': 9}}
        ]
    combination_method : str
        Método de combinación si indicators_combo es usado
        Opciones: 'AND', 'OR', 'MAJORITY', 'WEIGHTED', 'UNANIMOUS_LONG', 'UNANIMOUS_SHORT'
    
    Returns:
    --------
    dict : Métricas de performance o None si falla
    """
    # Calcular señales según tipo de estrategia
    if indicators_combo is not None:
        # Estrategia multi-indicador
        df = combine_indicator_signals(df, indicators_combo, combination_method)
    else:
        # Estrategia de indicador individual
        df = calculate_indicator_and_signals(df, indicator, params)
    
    if 'signal' not in df.columns or df['signal'].isna().all():
        return None
    
    # Obtener retornos futuros
    df['future_ret'] = df['returns'].shift(-1)
    
    # Filtrar por tipo de posición
    if position_type == 'long':
        mask = df['signal'] > 0
    elif position_type == 'short':
        mask = df['signal'] < 0
        df.loc[mask, 'future_ret'] = -df.loc[mask, 'future_ret']  # Invertir retornos para short
    else:  # both
        mask = df['signal'] != 0
        df.loc[df['signal'] < 0, 'future_ret'] = -df.loc[df['signal'] < 0, 'future_ret']
    
    trades = df[mask][['future_ret']].dropna()
    
    if len(trades) < 10:
        return None
    
    # Calcular métricas
    returns = trades['future_ret']
    cumulative_returns = (1 + returns).cumprod()
    
    total_return = cumulative_returns.iloc[-1] - 1
    n_trades = len(returns)
    
    # Hit rate
    hit_rate = (returns > 0).mean()
    
    # Sharpe Ratio (anualizado aproximado)
    sharpe = returns.mean() / returns.std() if returns.std() > 0 else 0
    
    # Max Drawdown
    running_max = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Win/Loss stats
    wins = returns[returns > 0]
    losses = returns[returns < 0]
    
    win_rate = len(wins) / n_trades if n_trades > 0 else 0
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    
    # Profit Factor
    gross_profit = wins.sum() if len(wins) > 0 else 0
    gross_loss = abs(losses.sum()) if len(losses) > 0 else 0
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0
    
    # Risk-Reward Ratio
    risk_reward = abs(avg_win / avg_loss) if avg_loss != 0 else 0
    
    # Calmar Ratio (retorno / max drawdown)
    calmar = total_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # Sortino Ratio (penaliza solo volatilidad bajista)
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() if len(downside_returns) > 0 else returns.std()
    sortino = returns.mean() / downside_std if downside_std > 0 else 0
    
    metrics = {
        'total_return': float(total_return),
        'n_trades': int(n_trades),
        'hit_rate': float(hit_rate),
        'win_rate': float(win_rate),
        'sharpe_ratio': float(sharpe),
        'sortino_ratio': float(sortino),
        'calmar_ratio': float(calmar),
        'max_drawdown': float(max_drawdown),
        'profit_factor': float(profit_factor),
        'avg_win': float(avg_win),
        'avg_loss': float(avg_loss),
        'risk_reward_ratio': float(risk_reward),
        'best_trade': float(returns.max()),
        'worst_trade': float(returns.min()),
        'avg_return_per_trade': float(returns.mean()),
        'volatility': float(returns.std())
    }
    
    return metrics


def strategy_grid_search(df, strategy_configs, use_mlflow=True, ticker='BTCUSDT', timeframe='1h',
                         experiment_name='momentum_trading_strategies'):
    """
    Grid Search para optimizar parámetros de estrategias de trading
    Calcula indicadores dinámicamente según configuración
    
    Parameters:
    -----------
    df : DataFrame
        Datos OHLCV básicos (open, high, low, close, volume, returns)
        NO necesita indicadores pre-calculados
    strategy_configs : list
        Lista de configuraciones de estrategia a probar
        Soporta estrategias individuales y combinadas
        
        Ejemplo Individual:
        [
            {
                'name': 'RSI_Reversal',
                'indicator': 'rsi',
                'params_grid': {
                    'period': [10, 14, 20, 30],
                    'overbought': [65, 70, 75],
                    'oversold': [25, 30, 35],
                    'position_type': ['long', 'short', 'both']
                }
            }
        ]
        
        Ejemplo Combinado:
        [
            {
                'name': 'RSI_MACD_Combo',
                'type': 'combo',  # <-- Indica estrategia combinada
                'indicators': [
                    {
                        'indicator': 'rsi',
                        'params_grid': {
                            'period': [14, 20],
                            'overbought': [70, 75],
                            'oversold': [25, 30]
                        }
                    },
                    {
                        'indicator': 'macd',
                        'params_grid': {
                            'fast': [12],
                            'slow': [26],
                            'signal': [9]
                        }
                    }
                ],
                'combination_methods': ['AND', 'OR', 'MAJORITY'],
                'position_types': ['long', 'both']
            }
        ]
    use_mlflow : bool
        Si True, registra experimentos en MLflow
    ticker : str
        Símbolo del activo (ej: 'BTCUSDT', 'ETHUSDT', 'SOLUSDT')
        Se normaliza a mayúsculas. Se usa como tag en MLflow para filtrar experimentos
        Default: 'BTCUSDT'
    timeframe : str
        Timeframe de los datos (ej: '1h', '4h', '1d', '15m')
        Se normaliza a minúsculas. Se usa como tag en MLflow para filtrar experimentos
        Default: '1h'
    experiment_name : str
        Nombre del experimento en MLflow
        Permite organizar experimentos por proyecto, estrategia, o contexto
        Default: 'momentum_trading_strategies'
        Default: '1h'
    
    Returns:
    --------
    DataFrame con resultados de todas las combinaciones
    Incluye columnas 'ticker' y 'timeframe' para análisis multi-asset
    """
    all_results = []
    
    # Normalizar ticker y timeframe
    ticker_normalized = ticker.upper()
    timeframe_normalized = timeframe.lower()
    
    print(f"\n🎯 Asset: {ticker_normalized} | Timeframe: {timeframe_normalized}")
    print(f"📊 Experimento MLflow: {experiment_name}")
    
    # Configurar MLflow
    if use_mlflow:
        try:
            import mlflow
            mlflow.set_tracking_uri("file:../mlruns")
            mlflow.set_experiment(experiment_name)
        except ImportError:
            print("⚠️  MLflow no disponible - tracking desactivado")
            use_mlflow = False
    
    # Calcular total de experimentos
    total_experiments = 0
    for config in strategy_configs:
        if config.get('type') == 'combo':
            # Estrategia combinada
            combo_experiments = 1
            for ind_config in config['indicators']:
                combo_experiments *= len(list(product(*ind_config['params_grid'].values())))
            combo_experiments *= len(config.get('combination_methods', ['AND']))
            combo_experiments *= len(config.get('position_types', ['long']))
            total_experiments += combo_experiments
        else:
            # Estrategia individual
            total_experiments += len(list(product(*config['params_grid'].values())))
    
    print(f"\n🔬 Iniciando Grid Search de Estrategias")
    print(f"   Total de experimentos: {total_experiments}")
    print(f"   MLflow tracking: {'✓ Activado' if use_mlflow else '✗ Desactivado'}")
    print("=" * 80)
    
    experiment_count = 0
    start_time = time.time()
    
    for strategy_config in strategy_configs:
        strategy_name = strategy_config['name']
        strategy_type = strategy_config.get('type', 'single')
        
        if strategy_type == 'combo':
            # ========== ESTRATEGIA COMBINADA ==========
            print(f"\n📊 Estrategia Combinada: {strategy_name}")
            print(f"   Indicadores: {[ind['indicator'] for ind in strategy_config['indicators']]}")
            
            # Generar todas las combinaciones de parámetros para cada indicador
            indicators_param_combos = []
            for ind_config in strategy_config['indicators']:
                param_names = list(ind_config['params_grid'].keys())
                param_values = list(ind_config['params_grid'].values())
                param_combinations = list(product(*param_values))
                indicators_param_combos.append({
                    'indicator': ind_config['indicator'],
                    'combos': [dict(zip(param_names, combo)) for combo in param_combinations]
                })
            
            # Generar producto cartesiano de todos los indicadores
            all_indicator_combos = list(product(*[ind['combos'] for ind in indicators_param_combos]))
            
            combination_methods = strategy_config.get('combination_methods', ['AND'])
            position_types = strategy_config.get('position_types', ['long'])
            
            total_combos = len(all_indicator_combos) * len(combination_methods) * len(position_types)
            print(f"   Combinaciones totales: {total_combos}")
            
            for indicator_params_set in all_indicator_combos:
                # Construir configuración de indicadores
                indicators_combo = []
                for idx, ind_params in enumerate(indicator_params_set):
                    indicators_combo.append({
                        'indicator': indicators_param_combos[idx]['indicator'],
                        'params': ind_params
                    })
                
                # Probar con diferentes métodos de combinación
                for combination_method in combination_methods:
                    for position_type in position_types:
                        experiment_count += 1
                        
                        # Ejecutar backtest combinado
                        metrics = backtest_strategy(
                            df=df,
                            indicator=None,
                            params={},
                            position_type=position_type,
                            indicators_combo=indicators_combo,
                            combination_method=combination_method
                        )
                        
                        if metrics is None:
                            continue
                        
                        # Preparar resultado
                        result = {
                            'ticker': ticker_normalized,
                            'timeframe': timeframe_normalized,
                            'strategy_name': strategy_name,
                            'strategy_type': 'combo',
                            'combination_method': combination_method,
                            'position_type': position_type,
                            'n_indicators': len(indicators_combo),
                            **metrics
                        }
                        
                        # Agregar parámetros de cada indicador
                        for idx, ind_combo in enumerate(indicators_combo):
                            indicator_name = ind_combo['indicator']
                            result[f'ind{idx+1}_name'] = indicator_name
                            for param_name, param_value in ind_combo['params'].items():
                                result[f'ind{idx+1}_{param_name}'] = param_value
                        
                        all_results.append(result)
                        
                        # Log en MLflow
                        if use_mlflow:
                            with mlflow.start_run(run_name=f"{strategy_name}_{experiment_count}"):
                                mlflow.log_param("strategy_name", strategy_name)
                                mlflow.log_param("strategy_type", "combo")
                                mlflow.log_param("combination_method", combination_method)
                                mlflow.log_param("position_type", position_type)
                                mlflow.log_param("n_indicators", len(indicators_combo))
                                
                                for idx, ind_combo in enumerate(indicators_combo):
                                    mlflow.log_param(f"ind{idx+1}_name", ind_combo['indicator'])
                                    for param_name, param_value in ind_combo['params'].items():
                                        mlflow.log_param(f"ind{idx+1}_{param_name}", param_value)
                                
                                for metric_name, metric_value in metrics.items():
                                    mlflow.log_metric(metric_name, metric_value)
                                
                                mlflow.set_tag("ticker", ticker_normalized)
                                mlflow.set_tag("timeframe", timeframe_normalized)
                                mlflow.set_tag("strategy_type", "combo")
                        
                        # Progreso
                        if experiment_count % 10 == 0:
                            elapsed = time.time() - start_time
                            avg_time = elapsed / experiment_count
                            remaining = (total_experiments - experiment_count) * avg_time
                            print(f"   [{experiment_count}/{total_experiments}] "
                                  f"Method: {combination_method} | "
                                  f"Sharpe: {metrics['sharpe_ratio']:.2f} | "
                                  f"ETA: {remaining:.0f}s")
        
        else:
            # ========== ESTRATEGIA INDIVIDUAL (comportamiento original) ==========
            indicator = strategy_config['indicator']
            params_grid = strategy_config['params_grid']
            
            # Generar todas las combinaciones de parámetros
            param_names = list(params_grid.keys())
            param_values = list(params_grid.values())
            param_combinations = list(product(*param_values))
            
            print(f"\n📊 Estrategia Individual: {strategy_name}")
            print(f"   Indicador: {indicator}")
            print(f"   Combinaciones: {len(param_combinations)}")
            
            for param_combo in param_combinations:
                experiment_count += 1
                params = dict(zip(param_names, param_combo))
                
                # Extraer position_type si existe
                position_type = params.pop('position_type', 'long')
                
                # Ejecutar backtest
                metrics = backtest_strategy(df, indicator, params, position_type)
                
                if metrics is None:
                    continue
                
                # Preparar resultado
                result = {
                    'ticker': ticker_normalized,
                    'timeframe': timeframe_normalized,
                    'strategy_name': strategy_name,
                    'strategy_type': 'single',
                    'indicator': indicator,
                    'position_type': position_type,
                    **params,
                    **metrics
                }
                
                all_results.append(result)
                
                # Log en MLflow
                if use_mlflow:
                    with mlflow.start_run(run_name=f"{strategy_name}_{experiment_count}"):
                        # Parámetros
                        mlflow.log_param("strategy_name", strategy_name)
                        mlflow.log_param("strategy_type", "single")
                        mlflow.log_param("indicator", indicator)
                        mlflow.log_param("position_type", position_type)
                        for param_name, param_value in params.items():
                            mlflow.log_param(param_name, param_value)
                        
                        # Métricas
                        for metric_name, metric_value in metrics.items():
                            mlflow.log_metric(metric_name, metric_value)
                        
                        # Tags
                        mlflow.set_tag("ticker", ticker_normalized)
                        mlflow.set_tag("timeframe", timeframe_normalized)
                        mlflow.set_tag("strategy_type", "single")
                
                # Progreso
                if experiment_count % 10 == 0:
                    elapsed = time.time() - start_time
                    avg_time = elapsed / experiment_count
                    remaining = (total_experiments - experiment_count) * avg_time
                    print(f"   [{experiment_count}/{total_experiments}] "
                          f"Sharpe: {metrics['sharpe_ratio']:.2f} | "
                          f"Win%: {metrics['win_rate']:.1%} | "
                          f"ETA: {remaining:.0f}s")
    
    elapsed = time.time() - start_time
    print(f"\n✓ Grid Search completado en {elapsed:.1f}s")
    print(f"  Experimentos exitosos: {len(all_results)}/{total_experiments}")
    
    results_df = pd.DataFrame(all_results)
    
    if not results_df.empty:
        # Ordenar por Sharpe Ratio
        results_df = results_df.sort_values('sharpe_ratio', ascending=False)
        
        # Mostrar top 5
        print(f"\n🏆 TOP 5 ESTRATEGIAS (por Sharpe Ratio):")
        print("=" * 80)
        for idx, row in results_df.head(5).iterrows():
            strategy_type = row.get('strategy_type', 'single')
            
            if strategy_type == 'combo':
                indicators_str = ', '.join([row.get(f'ind{i}_name', '?') for i in range(1, row.get('n_indicators', 0) + 1)])
                print(f"\n{idx+1}. {row['strategy_name']} (COMBO: {indicators_str})")
                print(f"   Method: {row['combination_method']}, Position: {row['position_type']}")
            else:
                print(f"\n{idx+1}. {row['strategy_name']} ({row.get('indicator', 'N/A')})")
                print(f"   Position: {row['position_type']}, Period: {row.get('period', 'N/A')}")
            
            print(f"   Sharpe: {row['sharpe_ratio']:.3f} | Win%: {row['win_rate']:.1%} | "
                  f"PF: {row['profit_factor']:.2f} | Return: {row['total_return']:.2%}")
    
    return results_df

print("✅ Funciones de Grid Search listas")

✅ Funciones de Grid Search listas


In [15]:
def visualize_grid_search_results(results_df, save_path='../data/img/grid_search_results.png'):
    """
    Visualiza resultados del Grid Search
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    if results_df.empty:
        print("⚠️  No hay resultados para visualizar")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Grid Search Results - Trading Strategies Optimization', fontsize=16, fontweight='bold')
    
    # 1. Sharpe Ratio Distribution
    axes[0, 0].hist(results_df['sharpe_ratio'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(results_df['sharpe_ratio'].median(), color='red', linestyle='--', label=f'Median: {results_df["sharpe_ratio"].median():.2f}')
    axes[0, 0].axvline(1.0, color='green', linestyle='--', alpha=0.5, label='Target: 1.0')
    axes[0, 0].set_xlabel('Sharpe Ratio')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Sharpe Ratio Distribution')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # 2. Win Rate vs Sharpe Ratio
    scatter = axes[0, 1].scatter(results_df['win_rate'], results_df['sharpe_ratio'], 
                                  c=results_df['profit_factor'], cmap='RdYlGn', 
                                  s=100, alpha=0.6, edgecolors='black')
    axes[0, 1].axhline(0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 1].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='50% Win Rate')
    axes[0, 1].set_xlabel('Win Rate')
    axes[0, 1].set_ylabel('Sharpe Ratio')
    axes[0, 1].set_title('Win Rate vs Sharpe Ratio (color = Profit Factor)')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    plt.colorbar(scatter, ax=axes[0, 1], label='Profit Factor')
    
    # 3. Total Return vs Max Drawdown
    axes[0, 2].scatter(results_df['max_drawdown'] * 100, results_df['total_return'] * 100,
                       c=results_df['sharpe_ratio'], cmap='viridis', s=100, alpha=0.6, edgecolors='black')
    axes[0, 2].axhline(0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 2].axvline(0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 2].set_xlabel('Max Drawdown (%)')
    axes[0, 2].set_ylabel('Total Return (%)')
    axes[0, 2].set_title('Return vs Risk')
    axes[0, 2].grid(alpha=0.3)
    
    # 4. Top 10 Strategies
    top_10 = results_df.nlargest(10, 'sharpe_ratio')
    strategy_labels = [f"{row['strategy_name'][:15]}_{row.get('period', 'N/A')}" 
                       for _, row in top_10.iterrows()]
    axes[1, 0].barh(range(len(top_10)), top_10['sharpe_ratio'], color='green', alpha=0.7, edgecolor='black')
    axes[1, 0].set_yticks(range(len(top_10)))
    axes[1, 0].set_yticklabels(strategy_labels, fontsize=8)
    axes[1, 0].set_xlabel('Sharpe Ratio')
    axes[1, 0].set_title('Top 10 Strategies by Sharpe')
    axes[1, 0].axvline(1.0, color='red', linestyle='--', alpha=0.5)
    axes[1, 0].invert_yaxis()
    axes[1, 0].grid(alpha=0.3, axis='x')
    
    # 5. Profit Factor Distribution by Position Type
    if 'position_type' in results_df.columns:
        position_types = results_df['position_type'].unique()
        for pos_type in position_types:
            data = results_df[results_df['position_type'] == pos_type]['profit_factor']
            axes[1, 1].hist(data, bins=20, alpha=0.6, label=pos_type, edgecolor='black')
        axes[1, 1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Break-even')
        axes[1, 1].set_xlabel('Profit Factor')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Profit Factor by Position Type')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
    
    # 6. Sharpe vs Number of Trades
    axes[1, 2].scatter(results_df['n_trades'], results_df['sharpe_ratio'],
                       c=results_df['win_rate'], cmap='plasma', s=100, alpha=0.6, edgecolors='black')
    axes[1, 2].set_xlabel('Number of Trades')
    axes[1, 2].set_ylabel('Sharpe Ratio')
    axes[1, 2].set_title('Sample Size vs Performance')
    axes[1, 2].axhline(0, color='black', linestyle='-', linewidth=0.5)
    axes[1, 2].grid(alpha=0.3)
    
    plt.tight_layout()
    
    # Guardar
    import os
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✓ Visualización guardada: {save_path}")
    
    plt.show()


def export_best_strategies(results_df, top_n=10, save_path='../data/stats/best_strategies.json'):
    """
    Exporta las mejores estrategias a JSON
    """
    import os
    
    if results_df.empty:
        print("⚠️  No hay resultados para exportar")
        return
    
    top_strategies = results_df.nlargest(top_n, 'sharpe_ratio')
    
    export_data = {
        'generated_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_strategies_tested': len(results_df),
        'top_strategies': top_strategies.to_dict('records')
    }
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    with open(save_path, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)
    
    print(f"✓ Mejores estrategias exportadas: {save_path}")
    print(f"  Top {top_n} estrategias guardadas")


print("✅ Funciones auxiliares de visualización y exportación definidas")

✅ Funciones auxiliares de visualización y exportación definidas


## 🎯 Grid Search: Parámetros Ticker y Timeframe

La función `strategy_grid_search()` ahora acepta parámetros para especificar el asset y timeframe:

```python
results = strategy_grid_search(
    df=data['1h'],
    strategy_configs=configs,
    use_mlflow=True,
    ticker='BTCUSDT',    # 🎯 Asset específico (default: 'BTCUSDT')
    timeframe='1h'       # 🎯 Timeframe específico (default: '1h')
)
```

### **¿Por qué es importante?**

1. **Multi-Asset Testing**: Testear la misma estrategia en BTC, ETH, SOL, etc.
2. **Multi-Timeframe Validation**: Comparar 1h vs 4h vs 1d
3. **MLflow Filtering**: Filtrar experimentos por `tags.ticker` o `tags.timeframe`
4. **Production Ready**: Identificar qué estrategias funcionan mejor en cada asset
5. **Result Organization**: DataFrame incluye columnas `ticker` y `timeframe` para análisis

### **Normalización Automática:**

- **Ticker**: Convertido a MAYÚSCULAS (`'btcusdt'` → `'BTCUSDT'`)
- **Timeframe**: Convertido a minúsculas (`'1H'` → `'1h'`)

### **Uso en MLflow:**

Los valores se guardan como tags en cada experimento:
```python
tags.ticker = "BTCUSDT"
tags.timeframe = "1h"
```

Filtrar en MLflow UI:
```
tags.ticker = "BTCUSDT" AND metrics.sharpe_ratio > 1.0
tags.ticker IN ("BTCUSDT", "ETHUSDT") AND tags.timeframe = "4h"
```

### **Análisis Multi-Asset:**

```python
# Consolidar resultados de múltiples assets
all_results = []
for ticker in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']:
    for tf in ['1h', '4h', '1d']:
        data = load_saved_data(ticker, [tf])
        results = strategy_grid_search(
            data[tf], configs, 
            ticker=ticker, timeframe=tf
        )
        all_results.append(results)

final_df = pd.concat(all_results)

# Análisis por asset
print(final_df.groupby('ticker')['sharpe_ratio'].max())

# Análisis por timeframe
print(final_df.groupby('timeframe')['sharpe_ratio'].max())
```

## 🔄 Grid Search Multi-Indicador

Ahora el grid search soporta estrategias multi-indicador:print("✅ Funciones de visualización y export listas")

### 📋 Ejemplo de Uso: Grid Search

A continuación se muestra cómo ejecutar el Grid Search para optimizar estrategias de trading.

### 🔗 Configuraciones para Estrategias Combinadas

Ahora el grid search soporta estrategias multi-indicador:

In [16]:
# Configuraciones de estrategias COMBINADAS para Grid Search
combo_strategy_configs = [
    # ========================================
    # COMBO 1: RSI + MACD (Reversión + Tendencia)
    # ========================================
    {
        'name': 'RSI_MACD_Combo',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'rsi',
                'params_grid': {
                    'period': [7, 14, 20],
                    'overbought': [65, 70, 75],
                    'oversold': [25, 30, 35]
                }
            },
            {
                'indicator': 'macd',
                'params_grid': {
                    'fast': [6, 12],
                    'slow': [24, 30],
                    'signal': [6, 9]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 2: Triple Oscilador (RSI + Stoch + Williams)
    # ========================================
    {
        'name': 'Triple_Oscillator',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'rsi',
                'params_grid': {
                    'period': [7, 14, 20],
                    'overbought': [65, 70, 75],
                    'oversold': [25, 30, 35]
                }
            },
            {
                'indicator': 'stoch',
                'params_grid': {
                    'k': [14],
                    'd': [3, 5, 7],
                    'overbought': [75, 80, 85],
                    'oversold': [15, 20, 25]
                }
            },
            {
                'indicator': 'willr',
                'params_grid': {
                    'period': [7, 14, 21],
                    'high_threshold': [-20],
                    'low_threshold': [-80]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 3: Trend Following Multi (MACD + ADX + Slope)
    # ========================================
    {
        'name': 'Trend_Multi',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'macd',
                'params_grid': {
                    'fast': [8, 12],
                    'slow': [21, 26],
                    'signal': [9]
                }
            },
            {
                'indicator': 'adx',
                'params_grid': {
                    'period': [14],
                    'threshold': [20, 25]
                }
            },
            {
                'indicator': 'slope',
                'params_grid': {
                    'period': [12, 24, 48],
                    'threshold': [0]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 4: Quality Filter (RSI + ADX)
    # ========================================
    # {
    #     'name': 'Quality_Filter',
    #     'type': 'combo',
    #     'indicators': [
    #         {
    #             'indicator': 'rsi',
    #             'params_grid': {
    #                 'period': [14, 20],
    #                 'overbought': [70, 75],
    #                 'oversold': [25, 30]
    #             }
    #         },
    #         {
    #             'indicator': 'adx',
    #             'params_grid': {
    #                 'period': [14],
    #                 'threshold': [20, 25, 30]
    #             }
    #         }
    #     ],
    #     'combination_methods': ['AND', 'UNANIMOUS_LONG'],
    #     'position_types': ['long', 'both']
    # }
]

# Calcular total de combinaciones
total_combo_experiments = 0
for config in combo_strategy_configs:
    combo_exp = 1
    for ind_config in config['indicators']:
        combo_exp *= len(list(product(*ind_config['params_grid'].values())))
    combo_exp *= len(config.get('combination_methods', ['AND']))
    combo_exp *= len(config.get('position_types', ['long']))
    total_combo_experiments += combo_exp
    print(f"\n📊 {config['name']}:")
    print(f"   Indicadores: {[ind['indicator'] for ind in config['indicators']]}")
    print(f"   Métodos: {config.get('combination_methods', ['AND'])}")
    print(f"   Combinaciones: {combo_exp}")

print(f"\n🚀 TOTAL COMBINACIONES (COMBOS): {total_combo_experiments}")
print(f"⏱️  Tiempo estimado (~2 seg/combo): ~{total_combo_experiments*2/60:.1f} minutos")


📊 RSI_MACD_Combo:
   Indicadores: ['rsi', 'macd']
   Métodos: ['AND']
   Combinaciones: 216

📊 Triple_Oscillator:
   Indicadores: ['rsi', 'stoch', 'willr']
   Métodos: ['AND']
   Combinaciones: 2187

📊 Trend_Multi:
   Indicadores: ['macd', 'adx', 'slope']
   Métodos: ['AND']
   Combinaciones: 24

🚀 TOTAL COMBINACIONES (COMBOS): 2427
⏱️  Tiempo estimado (~2 seg/combo): ~80.9 minutos


In [17]:

# ========================================
# EJECUTAR GRID SEARCH CON COMBOS
# ========================================

# Opción 1: Solo estrategias combinadas
# results_combos = strategy_grid_search(
#     df=data['1h'],
#     strategy_configs=combo_strategy_configs,
#     use_mlflow=True
# )

# Opción 2: Mezclar estrategias individuales y combinadas
# mixed_configs = strategy_configs[:3] + combo_strategy_configs[:2]  # 3 individuales + 2 combos
# results_mixed = strategy_grid_search(
#     df=data['1h'],
#     strategy_configs=mixed_configs,
#     use_mlflow=True
# )
data = calculate_returns_and_momentum(
    load_saved_data('btcusdt', ['1h'])['1h'],
    compute_indicators=False
)

# Opción 3: Solo un combo para prueba rápida
test_combo = [combo_strategy_configs[0]]  # RSI + MACD
results_test = strategy_grid_search(
    df=data,
    strategy_configs=test_combo,
    use_mlflow=True
)

print("✅ Grid search con estrategias combinadas configurado")
print("💡 Opciones:")
print("   1. Solo combos (combo_strategy_configs)")
print("   2. Mezclar individuales + combos (mixed_configs)")
print("   3. Test rápido (1 combo)")
print("\n⚠️  Los combos toman ~2x tiempo de individuales")

📂 Cargando datos guardados de btcusdt...
  ✓ 1h: 72107 velas | 2017-08-17 04:00:00 a 2025-11-12 22:00:00
    📄 Archivo: btcusdt_1h_20170817_20251112.parquet

✓ Datos cargados exitosamente desde disco

🎯 Asset: BTCUSDT | Timeframe: 1h

🔬 Iniciando Grid Search de Estrategias
   Total de experimentos: 216
   MLflow tracking: ✓ Activado

📊 Estrategia Combinada: RSI_MACD_Combo
   Indicadores: ['rsi', 'macd']
   Combinaciones totales: 216
  ✓ 1h: 72107 velas | 2017-08-17 04:00:00 a 2025-11-12 22:00:00
    📄 Archivo: btcusdt_1h_20170817_20251112.parquet

✓ Datos cargados exitosamente desde disco

🎯 Asset: BTCUSDT | Timeframe: 1h

🔬 Iniciando Grid Search de Estrategias
   Total de experimentos: 216
   MLflow tracking: ✓ Activado

📊 Estrategia Combinada: RSI_MACD_Combo
   Indicadores: ['rsi', 'macd']
   Combinaciones totales: 216
   [10/216] Method: AND | Sharpe: -0.04 | ETA: 187s
   [10/216] Method: AND | Sharpe: -0.04 | ETA: 187s
   [20/216] Method: AND | Sharpe: -0.02 | ETA: 127s
   [20/216]

### 📊 Análisis de Resultados: Individual vs Combo

**Cómo interpretar resultados de estrategias combinadas:**

#### **Columnas Específicas de Combos:**
- `strategy_type`: 'combo' vs 'single'
- `combination_method`: AND, OR, MAJORITY, etc.
- `n_indicators`: Número de indicadores combinados
- `ind1_name`, `ind2_name`, etc.: Nombre de cada indicador
- `ind1_period`, `ind2_threshold`, etc.: Parámetros de cada indicador

#### **Comparación Típica:**

```python
# Filtrar por tipo
singles = results[results['strategy_type'] == 'single']
combos = results[results['strategy_type'] == 'combo']

print("Individual - Promedio Sharpe:", singles['sharpe_ratio'].mean())
print("Combos - Promedio Sharpe:", combos['sharpe_ratio'].mean())

print("\nIndividual - Trades promedio:", singles['n_trades'].mean())
print("Combos - Trades promedio:", combos['n_trades'].mean())
```

#### **Preguntas Clave a Responder:**

1. **¿Los combos superan a individuales en Sharpe?**
   - Si sí: Validar que no sea overfitting
   - Si no: Mantener estrategias individuales

2. **¿Qué método de combinación funciona mejor?**
   - AND: Menos trades, mayor precisión
   - MAJORITY: Balance
   - OR: Más trades, menor precisión

3. **¿Cuántos indicadores son óptimos?**
   - 2 indicadores: Simple y robusto
   - 3 indicadores: Mayor filtrado
   - 4+: Riesgo de overf fitting

4. **¿Vale la pena la complejidad?**
   - Si Sharpe combo > 1.5x individual: SÍ
   - Si diferencia < 20%: NO, mantener simple

In [18]:
# Definir estrategias a probar - GRID SEARCH COMPLETO
strategy_configs = [
    # === OSCILLATORS - REVERSAL ===
    {
        'name': 'RSI_Reversal',
        'indicator': 'rsi',
        'params_grid': {
            'period': [10, 14, 20, 30],
            'overbought': [65, 70, 75, 80],
            'oversold': [20, 25, 30, 35],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'Williams_R_Reversal',
        'indicator': 'willr',
        'params_grid': {
            'period': [14, 21, 28],
            'high_threshold': [-15, -20, -25],
            'low_threshold': [-75, -80, -85],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'Stochastic_Reversal',
        'indicator': 'stoch',
        'params_grid': {
            'k': [14, 21],
            'd': [3, 5],
            'overbought': [75, 80, 85],
            'oversold': [15, 20, 25],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'CCI_Reversal',
        'indicator': 'cci',
        'params_grid': {
            'period': [14, 20, 30],
            'overbought': [80, 100, 120],
            'oversold': [-80, -100, -120],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'RSX_Reversal',
        'indicator': 'rsx',
        'params_grid': {
            'period': [10, 14, 20],
            'overbought': [65, 70, 75],
            'oversold': [25, 30, 35],
            'position_type': ['long', 'short', 'both']
        }
    },
    
    # === TREND INDICATORS ===
    {
        'name': 'MACD_Trend',
        'indicator': 'macd',
        'params_grid': {
            'fast': [8, 12, 16],
            'slow': [21, 26, 30],
            'signal': [7, 9, 11],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'ADX_Trend',
        'indicator': 'adx',
        'params_grid': {
            'period': [14, 20, 28],
            'threshold': [20, 25, 30],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'ER_Efficiency',
        'indicator': 'er',
        'params_grid': {
            'period': [10, 14, 20],
            'threshold': [0.2, 0.3, 0.4],
            'position_type': ['long', 'short', 'both']
        }
    },
    {
        'name': 'Slope_Trend',
        'indicator': 'slope',
        'params_grid': {
            'period': [10, 14, 20, 30],
            'threshold': [0, 0.001, 0.002],
            'position_type': ['long', 'short', 'both']
        }
    },
    
    # === MOMENTUM ADVANCED ===
    {
        'name': 'Inertia_Momentum',
        'indicator': 'inertia',
        'params_grid': {
            'period': [14, 20, 28],
            'high_threshold': [55, 60, 65],
            'low_threshold': [35, 40, 45],
            'position_type': ['long', 'short', 'both']
        }
    }
]

print("✅ Configuraciones de estrategias con GRID SEARCH COMPLETO:")
print(f"   Total de estrategias base: {len(strategy_configs)}")

total_combinations = 0
for config in strategy_configs:
    n_combos = len(list(product(*config['params_grid'].values())))
    total_combinations += n_combos
    print(f"\n📊 {config['name']}:")
    print(f"   Indicador: {config['indicator']}")
    print(f"   Parámetros a optimizar: {list(config['params_grid'].keys())}")
    print(f"   Combinaciones: {n_combos}")

print(f"\n🚀 TOTAL COMBINACIONES A PROBAR: {total_combinations}")
print(f"⏱️  Tiempo estimado (1 seg/combo): ~{total_combinations/60:.1f} minutos")

✅ Configuraciones de estrategias con GRID SEARCH COMPLETO:
   Total de estrategias base: 10

📊 RSI_Reversal:
   Indicador: rsi
   Parámetros a optimizar: ['period', 'overbought', 'oversold', 'position_type']
   Combinaciones: 192

📊 Williams_R_Reversal:
   Indicador: willr
   Parámetros a optimizar: ['period', 'high_threshold', 'low_threshold', 'position_type']
   Combinaciones: 81

📊 Stochastic_Reversal:
   Indicador: stoch
   Parámetros a optimizar: ['k', 'd', 'overbought', 'oversold', 'position_type']
   Combinaciones: 108

📊 CCI_Reversal:
   Indicador: cci
   Parámetros a optimizar: ['period', 'overbought', 'oversold', 'position_type']
   Combinaciones: 81

📊 RSX_Reversal:
   Indicador: rsx
   Parámetros a optimizar: ['period', 'overbought', 'oversold', 'position_type']
   Combinaciones: 81

📊 MACD_Trend:
   Indicador: macd
   Parámetros a optimizar: ['fast', 'slow', 'signal', 'position_type']
   Combinaciones: 81

📊 ADX_Trend:
   Indicador: adx
   Parámetros a optimizar: ['period'

In [19]:
# Ejecutar Grid Search con datos OHLCV básicos
# NOTA: Esta versión calcula indicadores y señales dinámicamente
# No necesitas pre-calcular nada, solo pasar los datos OHLCV básicos

# Ejemplo: Cargar o descargar datos primero
# data_btc = run_momentum_eda('BTC/USDT', ['1h'], use_cached=True)

# Ejecutar grid search (descomentar para ejecutar)
# results = strategy_grid_search(
#     df=data_btc['1h'],  # DataFrame con OHLCV y column 'returns'
#     strategy_configs=strategy_configs,
#     use_mlflow=True
# )

# # Visualizar resultados
# visualize_grid_search_results(results)

# # Exportar mejores estrategias
# export_best_strategies(results, top_n=10)

print("⚠️  Grid Search comentado - descomenta el código para ejecutar")
print("💡 Requisitos del DataFrame:")
print("   - Columnas necesarias: open, high, low, close, volume, returns")
print("   - Los indicadores se calculan automáticamente con los parámetros del grid")
print("   - Las señales se generan dinámicamente según thresholds configurados")

⚠️  Grid Search comentado - descomenta el código para ejecutar
💡 Requisitos del DataFrame:
   - Columnas necesarias: open, high, low, close, volume, returns
   - Los indicadores se calculan automáticamente con los parámetros del grid
   - Las señales se generan dinámicamente según thresholds configurados


### 🧪 Test Rápido - Subset Reducido

Para probar el sistema sin esperar mucho tiempo, puedes usar este subset reducido:

### 📚 Parámetros Soportados por Indicador

**OSCILLATORS (Reversal Strategies):**

| Indicador | Parámetros Disponibles | Valores por Defecto |
|-----------|------------------------|---------------------|
| **rsi** | period, overbought, oversold | 14, 70, 30 |
| **willr** | period, high_threshold, low_threshold | 14, -20, -80 |
| **stoch** | k, d, overbought, oversold | 14, 3, 80, 20 |
| **cci** | period, overbought, oversold | 14, 100, -100 |
| **rsx** | period, overbought, oversold | 14, 70, 30 |
| **cmo** | period, overbought, oversold | 14, 50, -50 |
| **uo** | overbought, oversold | 70, 30 |

**TREND INDICATORS:**

| Indicador | Parámetros Disponibles | Valores por Defecto |
|-----------|------------------------|---------------------|
| **macd** | fast, slow, signal | 12, 26, 9 |
| **adx** | period, threshold | 14, 25 |
| **er** | period, threshold | 14, 0.3 |
| **slope** | period, threshold | 14, 0 |
| **trix** | period, threshold | 14, 0 |
| **ao** | threshold | 0 |

**MOMENTUM ADVANCED:**

| Indicador | Parámetros Disponibles | Valores por Defecto |
|-----------|------------------------|---------------------|
| **inertia** | period, high_threshold, low_threshold | 14, 60, 40 |
| **bop** | threshold | 0 |

**COMÚN A TODOS:**
- `position_type`: ['long', 'short', 'both']

### ▶️ Ejemplo Completo - Workflow Recomendado

In [ ]:
# ========================================
# WORKFLOW COMPLETO - Grid Search
# ========================================

# PASO 1: Cargar datos (SOLO necesitas retornos básicos, no indicadores pre-calculados)
# Opción A: Descargar nuevos datos
# data = download_multi_timeframe_data('BTC/USDT', ['1h'], save_data=True)

# Opción B: Cargar desde cache
# data = load_saved_data('BTC/USDT', ['1h'])

# Opción C: Si tienes datos OHLCV, calcular solo retornos
# data_for_grid = calculate_returns_and_momentum(data['1h'], compute_indicators=False)
# Este modo es MÁS RÁPIDO porque no calcula los 36 indicadores innecesariamente

# PASO 2: Verificar que el DataFrame tenga las columnas necesarias
# print(data_for_grid.columns)
# Mínimo requerido: ['open', 'high', 'low', 'close', 'volume', 'returns']
# El grid search calculará indicadores dinámicamente según configuración

# PASO 3: Ejecutar Grid Search (descomentar para ejecutar)
# results = strategy_grid_search(
#     df=data_for_grid,  # DataFrame con OHLCV + returns
#     strategy_configs=strategy_configs,  # O test_configs para prueba rápida
#     use_mlflow=True
# )

# PASO 4: Analizar resultados
# print("\n🏆 TOP 10 ESTRATEGIAS:")
# print(results.nlargest(10, 'sharpe_ratio')[
#     ['strategy_name', 'indicator', 'period', 'position_type', 
#      'sharpe_ratio', 'win_rate', 'profit_factor', 'max_drawdown']
# ])

# PASO 5: Visualizar
# visualize_grid_search_results(results)

# PASO 6: Exportar mejores
# export_best_strategies(results, top_n=10)

# PASO 7: Ver en MLflow UI
# En terminal: mlflow ui --backend-store-uri file:../mlruns
# Luego abrir: http://localhost:5000

print("📝 Workflow completo documentado arriba")
print("💡 Descomenta línea por línea para ejecutar paso a paso")
print("\n⚡ TIP: Para grid search, usa compute_indicators=False")
print("   → Solo calcula retornos básicos (más rápido)")
print("   → Los indicadores se calculan dinámicamente con parámetros específicos")

---

## 🎯 Resumen del Sistema de Grid Search

### ✅ Características Implementadas

1. **Cálculo Dinámico de Indicadores**
   - Los indicadores se calculan on-the-fly con los parámetros del grid
   - No necesitas pre-calcular nada, solo pasar OHLCV básico
   - Soporta 17 indicadores diferentes

2. **Optimización de Períodos**
   - Prueba múltiples períodos: 10, 14, 20, 30, etc.
   - Cada período genera un indicador independiente

3. **Optimización de Thresholds**
   - Reversal: Prueba diferentes niveles de sobrecompra/sobreventa
   - Trend: Prueba diferentes umbrales de confirmación
   - Personalizable por indicador

4. **Backtesting Completo**
   - 16 métricas de performance
   - Soporte para long/short/both positions
   - Inversión correcta de retornos para shorts

5. **MLflow Integration**
   - Tracking automático de todos los experimentos
   - Comparación visual en UI
   - Exportación a CSV/JSON

### 📊 Indicadores Soportados

- **Oscillators (7)**: RSI, Williams %R, Stochastic, CCI, RSX, CMO, UO
- **Trend (6)**: MACD, ADX, ER, Slope, Trix, AO
- **Momentum (2)**: Inertia, BOP
- **Total**: 17 indicadores con parámetros configurables

### 🚀 Ventajas del Sistema

1. **Flexibilidad Total**: Prueba cualquier combinación periodo + threshold
2. **Eficiencia**: Cálculo optimizado, ~1 seg por combinación
3. **Reproducibilidad**: MLflow guarda todo automáticamente
4. **Escalabilidad**: Ejecuta overnight para grids masivos
5. **Análisis Completo**: 16 métricas por estrategia

### ⚡ Ejemplo Rápido

```python
# Configuración mínima
config = [{
    'name': 'RSI_Test',
    'indicator': 'rsi',
    'params_grid': {
        'period': [14, 20],
        'overbought': [70, 75],
        'oversold': [25, 30],
        'position_type': ['long', 'both']
    }
}]

# Ejecutar
results = strategy_grid_search(data['1h'], config, use_mlflow=True)
```

### 📈 Siguiente Nivel

Posibles extensiones futuras:
- Walk-forward validation (train/test split temporal)
- Multi-indicator combinators (AND/OR logic)
- Estrategias dinámicas (ajuste de posición según volatilidad)
- Optimización Bayesiana (más eficiente que grid search)
- Portfolio optimization (múltiples estrategias simultáneas)

---

---

## 🔗 Estrategias Multi-Indicador

### Combina Múltiples Indicadores en una Señal

Ahora puedes crear estrategias que combinan varios indicadores con diferentes métodos lógicos.

### 📋 Métodos de Combinación Disponibles

| Método | Descripción | Uso Típico |
|--------|-------------|------------|
| **AND** | Todas las señales deben coincidir (1 o -1) | Alta confianza, menos trades |
| **OR** | Al menos una señal != 0, suma señales | Mayor frecuencia, más trades |
| **MAJORITY** | Voto mayoritario (>50% indicadores) | Balance confianza/frecuencia |
| **WEIGHTED** | Promedio ponderado por pesos | Priorizar indicadores mejores |
| **UNANIMOUS_LONG** | Solo long si TODOS alcistas | Entradas muy conservadoras long |
| **UNANIMOUS_SHORT** | Solo short si TODOS bajistas | Entradas muy conservadoras short |

### 💡 Ejemplos de Uso

In [ ]:
# ========================================
# EJEMPLO 1: Estrategia RSI + MACD con AND
# ========================================
# Solo entra cuando AMBOS indicadores coinciden

combo_config_1 = [
    {
        'indicator': 'rsi',
        'params': {'period': 14, 'overbought': 70, 'oversold': 30}
    },
    {
        'indicator': 'macd',
        'params': {'fast': 12, 'slow': 26, 'signal': 9}
    }
]

# Backtest con combinación AND
# results_and = backtest_strategy(
#     df=data['1h'],
#     indicator=None,  # No se usa en combos
#     params={},       # No se usa en combos
#     position_type='both',
#     indicators_combo=combo_config_1,
#     combination_method='AND'
# )
# print("RSI + MACD (AND):", results_and['sharpe_ratio'], results_and['n_trades'])


# ========================================
# EJEMPLO 2: Estrategia Triple con MAJORITY
# ========================================
# RSI + Stochastic + Williams %R - voto mayoritario

combo_config_2 = [
    {
        'indicator': 'rsi',
        'params': {'period': 14, 'overbought': 70, 'oversold': 30}
    },
    {
        'indicator': 'stoch',
        'params': {'k': 14, 'd': 3, 'overbought': 80, 'oversold': 20}
    },
    {
        'indicator': 'willr',
        'params': {'period': 14, 'high_threshold': -20, 'low_threshold': -80}
    }
]

# results_majority = backtest_strategy(
#     df=data['1h'],
#     indicator=None,
#     params={},
#     position_type='long',
#     indicators_combo=combo_config_2,
#     combination_method='MAJORITY'
# )


# ========================================
# EJEMPLO 3: Estrategia Ponderada
# ========================================
# RSI (peso 2) + MACD (peso 3) + ADX (peso 1)

combo_config_3 = [
    {
        'indicator': 'rsi',
        'params': {'period': 14, 'overbought': 70, 'oversold': 30},
        'weight': 2.0
    },
    {
        'indicator': 'macd',
        'params': {'fast': 12, 'slow': 26, 'signal': 9},
        'weight': 3.0  # Mayor peso = más influencia
    },
    {
        'indicator': 'adx',
        'params': {'period': 14, 'threshold': 25},
        'weight': 1.0
    }
]

# results_weighted = backtest_strategy(
#     df=data['1h'],
#     indicator=None,
#     params={},
#     position_type='both',
#     indicators_combo=combo_config_3,
#     combination_method='WEIGHTED'
# )


print("✅ Ejemplos de estrategias multi-indicador configurados")
print("💡 Descomenta para ejecutar backtests de combinaciones")

### 🎯 Estrategias Recomendadas por Tipo

#### **Reversión a la Media (Mean Reversion)**
```python
# RSI + Williams %R + Stochastic (MAJORITY)
# Todos osciladores de sobrecompra/sobreventa
```
**Lógica**: 2 de 3 indicadores deben señalar extremo  
**Mejor en**: Mercados laterales con volatilidad

#### **Seguimiento de Tendencia (Trend Following)**
```python
# MACD + ADX + Slope (AND)
# Confirmar tendencia desde múltiples ángulos
```
**Lógica**: Todos deben confirmar dirección  
**Mejor en**: Mercados trending con momentum fuerte

#### **Filtro de Calidad (Quality Filter)**
```python
# RSI (reversión) + ADX (confirmar fuerza) (AND)
# Reversión confirmada por fuerza tendencial
```
**Lógica**: RSI señala entrada, ADX confirma que hay momentum  
**Mejor en**: Reducir falsos positivos en reversiones

#### **Agresivo Multi-Timeframe**
```python
# RSI_14 + RSI_20 + RSI_30 (MAJORITY)
# Mismo indicador, diferentes períodos
```
**Lógica**: Múltiples timeframes alineados  
**Mejor en**: Capturar señales robustas multi-horizonte

### ⚙️ Grid Search con Estrategias Multi-Indicador

Puedes hacer grid search sobre **métodos de combinación** además de parámetros:

In [ ]:
# Grid Search para encontrar mejor método de combinación
# Ejemplo: RSI + MACD con diferentes métodos

# Probar todos los métodos de combinación
combination_methods = ['AND', 'OR', 'MAJORITY', 'WEIGHTED', 'UNANIMOUS_LONG']

combo_results = []

# for method in combination_methods:
#     result = backtest_strategy(
#         df=data['1h'],
#         indicator=None,
#         params={},
#         position_type='both',
#         indicators_combo=[
#             {'indicator': 'rsi', 'params': {'period': 14, 'overbought': 70, 'oversold': 30}, 'weight': 2.0},
#             {'indicator': 'macd', 'params': {'fast': 12, 'slow': 26, 'signal': 9}, 'weight': 3.0}
#         ],
#         combination_method=method
#     )
#     
#     if result:
#         combo_results.append({
#             'method': method,
#             'sharpe': result['sharpe_ratio'],
#             'trades': result['n_trades'],
#             'win_rate': result['win_rate'],
#             'profit_factor': result['profit_factor']
#         })

# # Comparar resultados
# import pandas as pd
# comparison = pd.DataFrame(combo_results).sort_values('sharpe', ascending=False)
# print("\n🏆 Ranking de Métodos de Combinación:")
# print(comparison)

print("✅ Grid search multi-indicador configurado")
print("💡 Descomenta para comparar métodos de combinación")

### 📊 Comparación: Individual vs Combinado

**Ventajas de Estrategias Multi-Indicador:**

✅ **Reducción de Falsos Positivos**
- Indicador individual: 100 señales, 52% hit rate
- Combinado (AND): 40 señales, 65% hit rate
- Trade-off: Menos trades, mayor calidad

✅ **Mayor Sharpe Ratio**
- Filtrar señales débiles mejora risk-adjusted returns
- Ejemplo: RSI solo (Sharpe 0.8) → RSI+MACD (Sharpe 1.3)

✅ **Robustez en Diferentes Regímenes**
- Reversión (RSI) funciona lateral
- Tendencia (MACD) funciona trending
- Combinado (MAJORITY) se adapta mejor

❌ **Desventajas:**

- Menos oportunidades de trading
- Mayor complejidad computacional
- Posible sobreajuste si combinas demasiados

### 🎓 Mejores Prácticas

1. **Combinar indicadores de categorías diferentes**
   - ✅ RSI (oscilador) + MACD (tendencia)
   - ❌ RSI + Stochastic (ambos osciladores similares)

2. **Máximo 3-4 indicadores por estrategia**
   - Más allá es sobreajuste probable

3. **Usar WEIGHTED para priorizar indicadores probados**
   - Mayor peso al que tuvo mejor Sharpe individual

4. **Validar en out-of-sample**
   - Combos pueden overfittear más fácilmente que individuales

### 🌍 Multi-Asset / Multi-Timeframe Testing

Una vez configurados los parámetros `ticker` y `timeframe`, puedes testear la misma estrategia en múltiples assets:

In [ ]:
# EJEMPLO: Testear estrategia RSI en múltiples assets y timeframes
# 
# assets = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# timeframes = ['1h', '4h', '1d']
# 
# all_results = []
# 
# for ticker in assets:
#     for tf in timeframes:
#         print(f"\n🔄 Procesando {ticker} @ {tf}...")
#         
#         # Cargar datos
#         data = load_saved_data(ticker.replace('USDT', '').lower() + 'usdt', [tf])
#         data[tf] = calculate_returns_and_momentum(data[tf], compute_indicators=False)
#         
#         # Grid search con ticker/timeframe específicos
#         results = strategy_grid_search(
#             df=data[tf],
#             strategy_configs=test_configs,  # Usa configs definidas arriba
#             use_mlflow=True,
#             ticker=ticker,      # 🎯 Asset específico
#             timeframe=tf        # 🎯 Timeframe específico
#         )
#         
#         all_results.append(results)
# 
# # Consolidar todos los resultados
# final_df = pd.concat(all_results, ignore_index=True)
# 
# # Análisis por asset
# print("\n📊 Mejor Sharpe por Asset:")
# print(final_df.groupby('ticker')['sharpe_ratio'].max().sort_values(ascending=False))
# 
# print("\n📊 Mejor Sharpe por Timeframe:")
# print(final_df.groupby('timeframe')['sharpe_ratio'].max().sort_values(ascending=False))
# 
# # Top 5 global
# print("\n🏆 Top 5 Estrategias (todos los assets):")
# print(final_df.nlargest(5, 'sharpe_ratio')[['ticker', 'timeframe', 'strategy_name', 'sharpe_ratio', 'win_rate']])

print("✅ Ejemplo multi-asset/multi-timeframe configurado")
print("💡 Los resultados incluyen columnas 'ticker' y 'timeframe' para análisis")
print("💡 En MLflow UI puedes filtrar: tags.ticker = 'BTCUSDT' AND tags.timeframe = '1h'")

### 🖥️ Visualizar Experimentos en MLflow UI

Para explorar todos los experimentos en la interfaz web de MLflow:

```bash
# En terminal (PowerShell)
cd d:\py_projects\GammaNeutral\main
mlflow ui --backend-store-uri file:../mlruns
```

Luego abre tu navegador en: **http://localhost:5000**

**Funcionalidades de MLflow UI:**
- 📊 Comparar múltiples experimentos
- 🔍 Filtrar por métricas (ej: `metrics.sharpe_ratio > 1.0`)
- 📈 Gráficos paralelos de parámetros vs métricas
- 📥 Exportar resultados a CSV
- 🏆 Ordenar por cualquier métrica

### 🎯 Análisis de Resultados del Grid Search

**Métricas clave a analizar:**

1. **Sharpe Ratio** (> 1.0 excelente, > 2.0 excepcional)
   - Retorno ajustado por riesgo
   - Principal métrica de optimización

2. **Win Rate** (% de trades ganadores)
   - Reversal strategies: típicamente 45-55%
   - Trend strategies: típicamente 40-50%

3. **Profit Factor** (> 1.5 bueno, > 2.0 excelente)
   - Ganancia total / Pérdida total
   - Mide eficiencia de ganancias

4. **Max Drawdown** (< -20% aceptable, < -10% excelente)
   - Peor caída desde peak
   - Indica riesgo máximo

5. **Calmar Ratio** (retorno / |max_drawdown|)
   - Similar a Sharpe pero usa drawdown
   - > 0.5 bueno, > 1.0 excelente

**Estrategia de Análisis:**
1. Filtrar por Sharpe > 1.0
2. Verificar Max DD < -15%
3. Confirmar n_trades > 30 (muestra suficiente)
4. Comparar long vs short vs both
5. Validar en otros timeframes/activos

**Filtrado en MLflow UI por Asset/Timeframe:**
```python
# Ejemplos de filtros útiles:
tags.ticker = "BTCUSDT" AND tags.timeframe = "1h"
tags.ticker IN ("BTCUSDT", "ETHUSDT") AND metrics.sharpe_ratio > 1.0
tags.timeframe = "4h" AND metrics.win_rate > 0.50
```

---

## 🖥️ Visualizar Experimentos en MLflow UI

### Paso a Paso para Ver tus Experimentos

### 1️⃣ Verificar que MLflow guardó los experimentos

Primero asegúrate de que ejecutaste el grid search con `use_mlflow=True`:

```python
results = strategy_grid_search(
    df=data['1h'],
    strategy_configs=strategy_configs,
    use_mlflow=True  # <-- Debe estar en True
)
```

Los experimentos se guardan en: `d:\py_projects\GammaNeutral\mlruns\`

### 2️⃣ Iniciar MLflow UI desde Terminal

**Opción A: Desde PowerShell (Recomendado)**

Abre una nueva terminal PowerShell y ejecuta:

```powershell
# Navegar al directorio del proyecto
cd D:\py_projects\GammaNeutral\main

# Activar el ambiente conda
conda activate D:\py_projects\GammaNeutral\main\.conda

# Iniciar MLflow UI
mlflow ui --backend-store-uri file:../mlruns --host 127.0.0.1 --port 5000
```

**Opción B: Comando directo (si ya estás en el directorio correcto)**

```powershell
mlflow ui
```
MLflow automáticamente buscará la carpeta `mlruns` en el directorio padre.

**Salida esperada:**
```
[INFO] Starting gunicorn 20.1.0
[INFO] Listening at: http://127.0.0.1:5000
```

### 3️⃣ Abrir en el Navegador

Una vez que MLflow UI esté corriendo, abre tu navegador y ve a:

🌐 **http://localhost:5000**

o

🌐 **http://127.0.0.1:5000**

### 4️⃣ Navegar por la Interfaz de MLflow

#### **Pantalla Principal - Lista de Experimentos**

Verás tu experimento llamado: **`momentum_trading_strategies`**

Haz clic en él para ver todos los runs (experimentos individuales).

#### **Tabla de Runs**

Cada fila representa un experimento con:
- **Run Name**: `RSI_Reversal_1`, `MACD_Trend_2`, etc.
- **Métricas**: `sharpe_ratio`, `win_rate`, `profit_factor`, etc.
- **Parámetros**: `period`, `overbought`, `position_type`, etc.
- **Estado**: SUCCESS, FAILED, RUNNING
- **Fecha y duración**

### 5️⃣ Funcionalidades Clave de MLflow UI

#### **🔍 Filtrar Experimentos**

En la barra de búsqueda, puedes filtrar por:

```
# Sharpe mayor a 1.0
metrics.sharpe_ratio > 1.0

# Win rate mayor a 55%
metrics.win_rate > 0.55

# Estrategias RSI
params.indicator = "rsi"

# Posiciones long
params.position_type = "long"

# Estrategias combinadas
tags.strategy_type = "combo"

# Combinaciones múltiples
metrics.sharpe_ratio > 1.0 AND metrics.win_rate > 0.55 AND params.position_type = "long"
```

#### **📊 Comparar Runs**

1. Selecciona múltiples runs (checkbox a la izquierda)
2. Haz clic en **"Compare"**
3. Verás:
   - **Parallel Coordinates Plot**: Visualiza relación entre parámetros y métricas
   - **Scatter Plot**: Compara 2 métricas (ej: Sharpe vs Win Rate)
   - **Contour Plot**: Densidad de parámetros
   - **Tabla comparativa**: Todos los parámetros y métricas lado a lado

#### **📈 Gráficos de Parámetros**

1. Click en el botón **"Chart"** 
2. Crea gráficos personalizados:
   - **Bar Chart**: Comparar métricas entre runs
   - **Line Chart**: Evolución temporal
   - **Scatter Plot**: Relación entre 2 variables
   - **Parallel Coordinates**: Multi-dimensional

#### **📥 Exportar Datos**

1. Selecciona runs que te interesen
2. Click en **"Download CSV"**
3. Obtienes un archivo con todos los parámetros y métricas

#### **🏆 Ordenar por Métrica**

Haz clic en el header de cualquier columna para ordenar:
- Click en **"sharpe_ratio"** → Orden por Sharpe
- Click en **"n_trades"** → Orden por número de trades
- Click en **"profit_factor"** → Orden por profit factor

### 6️⃣ Ver Detalles de un Run Específico

Haz clic en cualquier Run Name para ver:

#### **📋 Parámetros**
Todos los parámetros de configuración:
```
strategy_name: RSI_Reversal
indicator: rsi
period: 14
overbought: 70
oversold: 30
position_type: long
```

#### **📊 Métricas**
Todos los resultados del backtest:
```
sharpe_ratio: 1.234
win_rate: 0.567
profit_factor: 1.89
total_return: 0.234
n_trades: 156
max_drawdown: -0.123
...
```

#### **🏷️ Tags**
Metadata adicional:
```
ticker: crypto
timeframe: intraday
strategy_type: single
```

#### **📝 Artifacts**
Archivos guardados (si se configuró):
- Modelos
- Gráficos
- Archivos de configuración

### 💡 Ejemplos Prácticos de Análisis

#### **Ejemplo 1: Encontrar mejores estrategias RSI**

1. En el filtro escribe: `params.indicator = "rsi"`
2. Ordena por columna `sharpe_ratio` (descendente)
3. Los primeros resultados son las mejores configuraciones RSI

#### **Ejemplo 2: Comparar estrategias combinadas vs individuales**

1. Filtrar individuales: `tags.strategy_type = "single"`
2. Exportar CSV
3. Filtrar combos: `tags.strategy_type = "combo"`
4. Exportar CSV
5. Comparar Sharpe promedio en Excel/Python

#### **Ejemplo 3: Análisis de combinación de métodos**

Para estrategias combinadas:
1. Filtrar: `tags.strategy_type = "combo"`
2. Crear gráfico de barras agrupando por `combination_method`
3. Ver cuál método (AND, OR, MAJORITY) da mejor Sharpe

#### **Ejemplo 4: Relación entre número de trades y Sharpe**

1. Seleccionar todos los runs
2. Click en "Compare"
3. Crear Scatter Plot: X=`n_trades`, Y=`sharpe_ratio`
4. Identificar sweet spot (muchos trades + alto Sharpe)

In [ ]:
# ========================================
# ACCEDER A MLFLOW DESDE PYTHON (Alternativa)
# ========================================

# Si prefieres analizar en Python en lugar de la UI:

import mlflow
import pandas as pd

# Configurar tracking URI
mlflow.set_tracking_uri("file:../mlruns")

# Obtener experimento
experiment = mlflow.get_experiment_by_name("momentum_trading_strategies")

if experiment:
    # Buscar todos los runs del experimento
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="metrics.sharpe_ratio > 1.0",  # Filtro opcional
        order_by=["metrics.sharpe_ratio DESC"],      # Ordenar por Sharpe
        max_results=100
    )
    
    print(f"✓ Se encontraron {len(runs)} runs")
    print("\n🏆 Top 5 por Sharpe Ratio:")
    print(runs[['params.strategy_name', 'params.indicator', 'metrics.sharpe_ratio', 
                'metrics.win_rate', 'metrics.n_trades']].head())
    
    # Análisis adicional
    # Filtrar por tipo
    # singles = runs[runs['tags.strategy_type'] == 'single']
    # combos = runs[runs['tags.strategy_type'] == 'combo']
    
    # print(f"\nPromedio Sharpe - Individuales: {singles['metrics.sharpe_ratio'].mean():.2f}")
    # print(f"Promedio Sharpe - Combinadas: {combos['metrics.sharpe_ratio'].mean():.2f}")
else:
    print("⚠️  Experimento no encontrado. Asegúrate de haber ejecutado grid search con use_mlflow=True")

print("\n💡 Tip: Para usar la UI visual, ejecuta en terminal:")
print("   mlflow ui --backend-store-uri file:../mlruns")

### ⚠️ Troubleshooting Común

#### **Problema 1: "ModuleNotFoundError: No module named 'mlflow'"**

**Solución:**
```powershell
conda activate D:\py_projects\GammaNeutral\main\.conda
pip install mlflow
```

#### **Problema 2: "No runs found"**

**Causas posibles:**
- No ejecutaste `strategy_grid_search()` con `use_mlflow=True`
- Los runs están en otra carpeta `mlruns`
- El experimento tiene otro nombre

**Verificar:**
```python
import os
print("¿Existe mlruns?", os.path.exists("../mlruns"))
```

#### **Problema 3: "Address already in use"**

Significa que MLflow ya está corriendo en ese puerto.

**Solución:**
```powershell
# Opción A: Usar otro puerto
mlflow ui --port 5001

# Opción B: Cerrar el proceso existente y reiniciar
```

#### **Problema 4: El navegador no abre automáticamente**

**Solución:** Abre manualmente en el navegador: http://localhost:5000

#### **Problema 5: No veo las columnas de métricas**

**Causa:** Necesitas agregar las columnas manualmente.

**Solución:** En MLflow UI:
1. Click en el ícono de configuración (⚙️) arriba a la derecha
2. Selecciona "Columns" → "Add metric"
3. Agrega: `sharpe_ratio`, `win_rate`, `profit_factor`, etc.

### 📚 Resumen - Comandos Rápidos

```powershell
# 1. Navegar al proyecto
cd D:\py_projects\GammaNeutral\main

# 2. Activar ambiente
conda activate D:\py_projects\GammaNeutral\main\.conda

# 3. Iniciar MLflow UI
mlflow ui --backend-store-uri file:../mlruns

# 4. Abrir navegador
# Ve a: http://localhost:5000

# 5. Para detener MLflow
# Presiona Ctrl+C en la terminal
```

### 🎯 Workflow Completo

```mermaid
graph LR
    A[Ejecutar Grid Search] --> B[use_mlflow=True]
    B --> C[Datos guardados en mlruns/]
    C --> D[Abrir Terminal]
    D --> E[mlflow ui]
    E --> F[Abrir localhost:5000]
    F --> G[Analizar Resultados]
    G --> H[Exportar mejores]
```

### 🚀 Siguiente Paso

Una vez que hayas identificado las mejores estrategias en MLflow UI:

1. **Exportar a CSV** las mejores configuraciones
2. **Validar out-of-sample** en datos diferentes
3. **Implementar en producción** las estrategias ganadoras
4. **Monitorear performance** con MLflow tracking continuo

---

## 📚 Resumen: Workflow Completo de Optimización

### **1. Cargar Datos con Asset/Timeframe Específico**
```python
# Cargar datos de BTC/USDT en 1h
market_data = load_saved_data('btcusdt', ['1h'])
market_data['1h'] = calculate_returns_and_momentum(
    market_data['1h'], 
    compute_indicators=False  # Solo retornos para grid search
)
```

### **2. Configurar Estrategias (Individual o Combo)**

**Estrategia Individual:**
```python
rsi_config = {
    'name': 'RSI_Strategy',
    'indicator': 'rsi',
    'params_grid': {
        'period': [10, 14, 20],
        'overbought': [70, 75, 80],
        'oversold': [20, 25, 30],
        'position_type': ['long', 'short', 'both']
    }
}
```

**Estrategia Combinada:**
```python
combo_config = {
    'name': 'RSI_MACD_Combo',
    'type': 'combo',
    'indicators': [
        {
            'indicator': 'rsi',
            'params_grid': {'period': [14], 'overbought': [70], 'oversold': [30]}
        },
        {
            'indicator': 'macd',
            'params_grid': {'fast': [12], 'slow': [26], 'signal': [9]}
        }
    ],
    'combination_methods': ['AND', 'MAJORITY'],
    'position_types': ['both']
}
```

### **3. Ejecutar Grid Search con Ticker/Timeframe**
```python
results = strategy_grid_search(
    df=market_data['1h'],
    strategy_configs=[rsi_config, combo_config],
    use_mlflow=True,
    ticker='BTCUSDT',  # 🎯 Asset específico
    timeframe='1h'     # 🎯 Timeframe específico
)

# Ver mejores estrategias
print(results.nlargest(10, 'sharpe_ratio')[
    ['ticker', 'timeframe', 'strategy_name', 'sharpe_ratio', 'win_rate', 'max_drawdown']
])
```

### **4. Multi-Asset/Multi-Timeframe Testing**
```python
all_results = []
for ticker in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']:
    for tf in ['1h', '4h', '1d']:
        data = load_saved_data(ticker.replace('USDT','').lower()+'usdt', [tf])
        data[tf] = calculate_returns_and_momentum(data[tf], compute_indicators=False)
        
        results = strategy_grid_search(
            data[tf], configs,
            ticker=ticker, timeframe=tf
        )
        all_results.append(results)

final_df = pd.concat(all_results)

# Análisis por asset
final_df.groupby('ticker')['sharpe_ratio'].max()
```

### **5. Analizar en MLflow UI**
```bash
# En PowerShell
cd D:\py_projects\GammaNeutral\main
mlflow ui --backend-store-uri file:../mlruns
# Abrir: http://localhost:5000
```

**Filtros útiles en MLflow:**
- `tags.ticker = "BTCUSDT" AND tags.timeframe = "1h"`
- `tags.ticker IN ("BTCUSDT", "ETHUSDT") AND metrics.sharpe_ratio > 1.0`
- `tags.timeframe = "4h" AND metrics.win_rate > 0.50`

### **6. Visualizar y Exportar**
```python
# Visualización
visualize_grid_search_results(results)

# Exportar mejores
export_best_strategies(results, top_n=10)
```

### **📊 Columnas Clave en Results DataFrame:**

**Metadatos:**
- `ticker`: Asset (ej: 'BTCUSDT')
- `timeframe`: Periodo (ej: '1h')
- `strategy_name`: Nombre de la estrategia
- `strategy_type`: 'single' o 'combo'

**Parámetros:**
- `indicator`: Nombre del indicador
- `period`, `overbought`, `oversold`, etc.: Parámetros específicos
- `combination_method`: Método de combinación (si es combo)

**Métricas de Performance:**
- `sharpe_ratio`: Retorno ajustado por riesgo
- `win_rate`: % de trades ganadores
- `profit_factor`: Total gain / Total loss
- `max_drawdown`: Peor caída desde peak
- `total_return`: Retorno acumulado
- `n_trades`: Número de operaciones

### **✅ Checklist de Validación:**

- [ ] Sharpe Ratio > 1.0
- [ ] Max Drawdown < -15%
- [ ] n_trades > 30 (muestra suficiente)
- [ ] Profit Factor > 1.5
- [ ] Validado en múltiples assets
- [ ] Validado en múltiples timeframes
- [ ] Probado out-of-sample

---

**🎯 Tu sistema ahora puede:**
1. ✅ Calcular 36 indicadores técnicos con pandas-ta
2. ✅ Combinar múltiples indicadores (6 métodos)
3. ✅ Optimizar hiperparámetros con Grid Search
4. ✅ Trackear experimentos en MLflow con asset/timeframe
5. ✅ Testear en múltiples assets y timeframes
6. ✅ Filtrar y comparar resultados por ticker/timeframe
7. ✅ Visualizar y exportar mejores estrategias

**📈 Próximos pasos sugeridos:**
- Walk-forward validation
- Monte Carlo simulation
- Parameter importance analysis
- Real-time data integration

---

## 🚀 Ejemplo Ejecutable Completo

A continuación un ejemplo completo que puedes ejecutar directamente:

In [29]:
# Configuraciones de estrategias COMBINADAS para Grid Search
combo_strategy_configs = [
    {
        'name': 'RSI_simple',
        'type': 'single',
        'indicator': 'rsi',
        'params_grid': {
            'period': [14],
            'overbought': [70],
            'oversold': [30],
            'position_type': ['both']
            }
    },
    {
        'name': 'MACD_simple',
        'type': 'single',
        'indicator': 'macd',
        'params_grid': {
            'fast': [12],
            'slow': [26],
            'signal': [9],
            'position_type': ['both']
            }
    },
    {
        'name': 'willr_simple',
        'type': 'single',
        'indicator': 'willr',
        'params_grid': {
            'period': [14],
            'high_threshold': [-20],
            'low_threshold': [-80],
            'position_type': ['both']
            }
    },
    {
        'name': 'stoch_simple',
        'type': 'single',
        'indicator': 'stoch',
        'params_grid': {
            'k': [14],
            'd': [5],
            'overbought': [80],
            'oversold': [20],
            'position_type': ['both']
            }
    },
    {
        'name': 'adx_simple',
        'type': 'single',
        'indicator': 'adx',
        'params_grid': {
            'period': [14],
            'threshold': [20, 25],
            'position_type': ['both']
            }
    },
    {
        'name': 'slope_simple',
        'type': 'single',
        'indicator': 'slope',
        'params_grid': {
            'period': [24],
            'threshold': [0],
            'position_type': ['both']
            }
    },
    # ========================================
    # COMBO 1: RSI + MACD (Reversión + Tendencia)
    # ========================================
    {
        'name': 'RSI_MACD_Combo',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'rsi',
                'params_grid': {
                    'period': [14],
                    'overbought': [70],
                    'oversold': [30]
                }
            },
            {
                'indicator': 'macd',
                'params_grid': {
                    'fast': [12],
                    'slow': [26],
                    'signal': [9]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 2: Triple Oscilador (RSI + Stoch + Williams)
    # ========================================
    {
        'name': 'Triple_Oscillator',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'rsi',
                'params_grid': {
                    'period': [14],
                    'overbought': [70],
                    'oversold': [30]
                }
            },
            {
                'indicator': 'stoch',
                'params_grid': {
                    'k': [14],
                    'd': [7],
                    'overbought': [80],
                    'oversold': [20]
                }
            },
            {
                'indicator': 'willr',
                'params_grid': {
                    'period': [14],
                    'high_threshold': [-20],
                    'low_threshold': [-80]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 3: Trend Following Multi (MACD + ADX + Slope)
    # ========================================
    {
        'name': 'Trend_Multi',
        'type': 'combo',
        'indicators': [
            {
                'indicator': 'macd',
                'params_grid': {
                    'fast': [12],
                    'slow': [26],
                    'signal': [9]
                }
            },
            {
                'indicator': 'adx',
                'params_grid': {
                    'period': [14],
                    'threshold': [20, 25]
                }
            },
            {
                'indicator': 'slope',
                'params_grid': {
                    'period': [12, 24, 48],
                    'threshold': [0]
                }
            }
        ],
        'combination_methods': ['AND'],
        'position_types': ['both']
    },
    
    # ========================================
    # COMBO 4: Quality Filter (RSI + ADX)
    # ========================================
    # {
    #     'name': 'Quality_Filter',
    #     'type': 'combo',
    #     'indicators': [
    #         {
    #             'indicator': 'rsi',
    #             'params_grid': {
    #                 'period': [14, 20],
    #                 'overbought': [70, 75],
    #                 'oversold': [25, 30]
    #             }
    #         },
    #         {
    #             'indicator': 'adx',
    #             'params_grid': {
    #                 'period': [14],
    #                 'threshold': [20, 25, 30]
    #             }
    #         }
    #     ],
    #     'combination_methods': ['AND', 'UNANIMOUS_LONG'],
    #     'position_types': ['long', 'both']
    # }
]

# Calcular total de combinaciones
total_combo_experiments = 0
for config in combo_strategy_configs:
    combo_exp = 1
    
    # Diferenciar entre estrategias simples y combo
    if config['type'] == 'single':
        # Para estrategias simples, usar 'params_grid' directamente
        combo_exp = len(list(product(*config['params_grid'].values())))
        print(f"\n📊 {config['name']} (single):")
        print(f"   Indicador: {config['indicator']}")
        print(f"   Combinaciones: {combo_exp}")
    else:
        # Para estrategias combo, iterar sobre 'indicators'
        for ind_config in config['indicators']:
            combo_exp *= len(list(product(*ind_config['params_grid'].values())))
        combo_exp *= len(config.get('combination_methods', ['AND']))
        combo_exp *= len(config.get('position_types', ['long']))
        print(f"\n📊 {config['name']} (combo):")
        print(f"   Indicadores: {[ind['indicator'] for ind in config['indicators']]}")
        print(f"   Métodos: {config.get('combination_methods', ['AND'])}")
        print(f"   Combinaciones: {combo_exp}")
    
    total_combo_experiments += combo_exp

print(f"\n🚀 TOTAL COMBINACIONES (COMBOS): {total_combo_experiments}")
print(f"⏱️  Tiempo estimado (~2 seg/combo): ~{total_combo_experiments*2/60:.1f} minutos")


📊 RSI_simple (single):
   Indicador: rsi
   Combinaciones: 1

📊 MACD_simple (single):
   Indicador: macd
   Combinaciones: 1

📊 willr_simple (single):
   Indicador: willr
   Combinaciones: 1

📊 stoch_simple (single):
   Indicador: stoch
   Combinaciones: 1

📊 adx_simple (single):
   Indicador: adx
   Combinaciones: 2

📊 slope_simple (single):
   Indicador: slope
   Combinaciones: 1

📊 RSI_MACD_Combo (combo):
   Indicadores: ['rsi', 'macd']
   Métodos: ['AND']
   Combinaciones: 1

📊 Triple_Oscillator (combo):
   Indicadores: ['rsi', 'stoch', 'willr']
   Métodos: ['AND']
   Combinaciones: 1

📊 Trend_Multi (combo):
   Indicadores: ['macd', 'adx', 'slope']
   Métodos: ['AND']
   Combinaciones: 6

🚀 TOTAL COMBINACIONES (COMBOS): 15
⏱️  Tiempo estimado (~2 seg/combo): ~0.5 minutos


In [31]:
# =============================================================================
# EJEMPLO COMPLETO: Grid Search con Asset/Timeframe Específicos
# =============================================================================

# Lista de tickers y timeframes a testear
tickers = [
    'BTCUSDT',
    'ETHUSDT',
    #'SOLUSDT',
    #'LTCUSDT',
    #'UNIUSDT',
    #'ZECUSDT'
    ]
timeframes = [
    '1h',
    #'4h',
    #'1d'
    ]

all_results = []

for ticker in tickers:
    for tf in timeframes:
        print(f"\n🔄 Procesando {ticker} @ {tf}...")
        
        try:
            # Cargar datos
            market_data = load_saved_data(ticker.lower(), [tf])
            market_data[tf] = calculate_returns_and_momentum(
                market_data[tf], 
                compute_indicators=False  # Solo retornos básicos
            )
            
            # Grid search con ticker/timeframe específicos
            results = strategy_grid_search(
                df=market_data[tf],
                strategy_configs=combo_strategy_configs,  # Usa configs definidas arriba
                use_mlflow=True,
                ticker=ticker,      # 🎯 Asset específico
                timeframe=tf        # 🎯 Timeframe específico
            )
            
            # Acumular resultados
            all_results.append(results)
            
            # Mostrar top 3 de este asset/timeframe
            print("\n🏆 TOP 3 ESTRATEGIAS:")
            print(results.nlargest(3, 'sharpe_ratio')[[
                'strategy_name', 'sharpe_ratio', 'win_rate', 'n_trades'
            ]])
            
        except Exception as e:
            print(f"❌ Error procesando {ticker} @ {tf}: {str(e)}")
            continue

# Consolidar todos los resultados
final_df = pd.concat(all_results, ignore_index=True)

# Análisis global
print("\n📊 ANÁLISIS GLOBAL:")
print(f"Total estrategias testeadas: {len(final_df)}")
print(f"\nMejor Sharpe por Asset:")
print(final_df.groupby('ticker')['sharpe_ratio'].max().sort_values(ascending=False))
print(f"\nMejor Sharpe por Timeframe:")
print(final_df.groupby('timeframe')['sharpe_ratio'].max().sort_values(ascending=False))

# Top 10 global
print("\n🏆 TOP 10 ESTRATEGIAS GLOBALES:")
print(final_df.nlargest(10, 'sharpe_ratio')[[
    'ticker', 'timeframe', 'strategy_name', 'sharpe_ratio', 'win_rate', 'profit_factor', 'n_trades'
]])

print("\n✅ GRID SEARCH COMPLETADO")
print("💡 Para ver detalles en MLflow UI:")
print("   1. Abre terminal PowerShell")
print("   2. cd D:\\py_projects\\GammaNeutral\\main")
print("   3. mlflow ui --backend-store-uri file:../mlruns")
print("   4. Abre: http://localhost:5000")
print("   5. Filtra por asset/timeframe:")
print("      tags.ticker = 'BTCUSDT' AND tags.timeframe = '1h'")
print("      tags.ticker IN ('BTCUSDT', 'ETHUSDT') AND metrics.sharpe_ratio > 1.0")


🔄 Procesando BTCUSDT @ 1h...
📂 Cargando datos guardados de btcusdt...
  ✓ 1h: 72107 velas | 2017-08-17 04:00:00 a 2025-11-12 22:00:00
    📄 Archivo: btcusdt_1h_20170817_20251112.parquet

✓ Datos cargados exitosamente desde disco

🎯 Asset: BTCUSDT | Timeframe: 1h

🔬 Iniciando Grid Search de Estrategias
   Total de experimentos: 15
   MLflow tracking: ✓ Activado

📊 Estrategia Individual: RSI_simple
   Indicador: rsi
   Combinaciones: 1
  ✓ 1h: 72107 velas | 2017-08-17 04:00:00 a 2025-11-12 22:00:00
    📄 Archivo: btcusdt_1h_20170817_20251112.parquet

✓ Datos cargados exitosamente desde disco

🎯 Asset: BTCUSDT | Timeframe: 1h

🔬 Iniciando Grid Search de Estrategias
   Total de experimentos: 15
   MLflow tracking: ✓ Activado

📊 Estrategia Individual: RSI_simple
   Indicador: rsi
   Combinaciones: 1

📊 Estrategia Individual: MACD_simple
   Indicador: macd
   Combinaciones: 1

📊 Estrategia Individual: MACD_simple
   Indicador: macd
   Combinaciones: 1

📊 Estrategia Individual: willr_simple


ValueError: No objects to concatenate